# GigaHands · 두 손 3D Transformer 학습

현재 프로젝트의 **시간(초) 기준 이벤트 모델**을 포함한 독립 실행 노트북입니다. 입력은 실제 도착한 카메라 프레임(이벤트) 목록이고, 임의의 query 시각마다 `[2,21,3]`을 출력합니다. 설정 셀의 `ARCH`로 모델을 고릅니다: 기본 `'lite'`(HandLite: 카메라당 최근 이벤트 8개만 쓰는 고정 크기 모델, ncnn/ONNX로 라즈베리 파이 배포) 또는 `'transformer'`(HandTransformer: 비교 기준). 카메라 보정은 장기 기억 없이 최근 약 0.5초의 이벤트에서만 추정합니다. 별도 저장소 복제가 필요하지 않습니다. 모든 범위가 초 단위라 출력 fps에 묶이지 않으며 실제 파라미터 수는 아래 확인 셀에서 표시합니다. 카메라 보정용 보조 정답 `[3,6]`은 NPZ의 실제·명목 카메라 값에서 계산해 손실에만 사용합니다.

## 먼저 할 일
1. Colab에서 이 `.ipynb`를 엽니다.
2. **런타임 → 런타임 유형 변경 → GPU**를 선택합니다.
3. PC의 `exports/gigahands_balanced_no_roll_full.zip`을 Google Drive의 `MyDrive/GigaHands/`에 올립니다. ZIP 안에는 `manifest.jsonl`, `train/`, `val/`가 있어야 합니다. 원본 GigaHands tar.gz가 아닙니다.
4. 아래 설정 셀의 경로와 실행 이름을 확인한 뒤 위에서부터 실행합니다.

데이터: 원본 클립 12,774개, 학습 10,988개/검증 1,786개, 참가자 45명/11명. ZIP 약 10.77GB, 압축 해제 후 약 11.35GB입니다. Colab GPU 종류·사용 시간은 가용성에 따라 달라집니다. Drive에 많은 작은 파일을 직접 반복 읽는 대신 ZIP을 런타임 로컬로 복사합니다. [Colab 공식 FAQ](https://research.google.com/colaboratory/faq.html)

기존 이전 아키텍처의 smoke 가중치는 사용하지 않습니다. 새 모델을 처음부터 학습하며, 중단 후에는 이 노트북이 저장한 같은 실행의 가중치만 재개합니다.


In [ ]:
# 1. 환경 확인 — Colab에 설치된 CUDA용 PyTorch를 그대로 사용합니다.
import sys, os, json, hashlib, shutil, subprocess, importlib.util
from pathlib import Path, PurePosixPath
import torch

assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 5), 'PyTorch 2.5 이상이 필요합니다.'
assert torch.cuda.is_available(), 'Colab 런타임 유형을 GPU로 바꾸고 다시 실행하세요.'
missing = [name for name in ('numpy', 'matplotlib', 'tqdm') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import numpy as np
from tqdm.auto import tqdm
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
WORK_DIR = Path('/content/gigahands_training')
WORK_DIR.mkdir(exist_ok=True)
(WORK_DIR / 'hand_tracking').mkdir(exist_ok=True)
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))


In [ ]:
# 2. Drive 연결
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3. 사용자 설정
DRIVE_ZIP = Path('/content/drive/MyDrive/GigaHands/gigahands_balanced_no_roll_full.zip')
DRIVE_RUNS = Path('/content/drive/MyDrive/GigaHands/runs')
RUN_NAME = 'event_lite_no_roll_v1'  # 이전 노트북의 실행과 소스가 달라 새 이름으로 학습
EPOCHS = 30
BATCH_SIZE = 16                   # GPU 메모리 부족이면 새 실행 이름으로 8 또는 4
LEARNING_RATE = 3e-4
WINDOWS_PER_CLIP = 16             # epoch마다 클립당 무작위 구간; 0이면 전체
VAL_WINDOWS_PER_CLIP = 16         # 매 epoch 고정 검증 구간; 0이면 전체
WORKERS = 2
SEED = 42
RIG_ROTATE_DEG = 45.0             # 현재 수정 모델의 학습 증강 기본값
RIG_SHIFT_WORLD = 0.25            # 1월드=30cm → 축별 최대 7.5cm 이동
# 'lite': HandLite — 고정 크기, ncnn/ONNX 배포용(라즈베리 파이 실시간 목표)
# 'transformer': HandTransformer — 이벤트 전체 attention(비교 기준)
ARCH = 'lite'
MODEL_OPTIONS = {
    'lite': {'--dim': 64, '--heads': 4, '--blocks': 2, '--slots-per-camera': 8,   # 카메라당 최근 이벤트 8개
             '--event-span-s': 0.5, '--anchor-lookback-s': 0.2},
    'transformer': {'--dim': 96, '--heads': 4, '--blocks': 2, '--encoder-layers': 1,
                    '--event-span-s': 0.4, '--calibration-span-s': 0.5, '--anchor-lookback-s': 0.2},
}[ARCH]
DROPOUT = 0.1
CALIBRATION_WEIGHT = 0.01         # 카메라 보정 보조 정답의 손실 가중치
MODEL_ARGS = ['--arch', ARCH, '--dropout', DROPOUT] + [value for item in MODEL_OPTIONS.items() for value in item]

assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in ('.', '..')
assert EPOCHS > 0 and BATCH_SIZE > 0 and WORKERS >= 0
assert LEARNING_RATE > 0 and WINDOWS_PER_CLIP >= 0 and VAL_WINDOWS_PER_CLIP >= 0
assert DRIVE_ZIP.is_file(), f'Drive에 데이터 ZIP을 올리고 경로를 확인하세요: {DRIVE_ZIP}'
LOCAL_ZIP = Path('/content/gigahands_balanced_no_roll_full.zip')
EXTRACT_DIR = Path('/content/gigahands_dataset')
EXPERIMENT_DIR = DRIVE_RUNS / RUN_NAME
RUN_DIR = EXPERIMENT_DIR / 'checkpoints'
print('학습 결과:', RUN_DIR)


## 데이터 복사 및 검사
처음 실행할 때 데이터 ZIP을 복사·해제합니다. 연결이 끊겨 런타임 로컬 파일이 사라지면 이 셀을 다시 실행하세요. Drive의 원본 ZIP과 체크포인트는 보존됩니다. 진행 막대가 표시됩니다.


In [ ]:
# 4. ZIP 로컬 복사 + 안전한 압축 해제 (기존 부분 해제는 다시 완료)
import zipfile, stat
EXPECTED_MANIFEST_SHA256 = 'd2499b14ba63d935d429f0ae9555f3fc61818d3b72acc680c61fccede04cc680'
zip_size = DRIVE_ZIP.stat().st_size
if not LOCAL_ZIP.is_file() or LOCAL_ZIP.stat().st_size != zip_size:
    assert shutil.disk_usage('/content').free > zip_size + 11354029117 + 1_000_000_000, '로컬 디스크 공간이 부족합니다.'
    partial = LOCAL_ZIP.with_suffix('.zip.part')
    with DRIVE_ZIP.open('rb') as src, partial.open('wb') as dst, tqdm(total=zip_size, unit='B', unit_scale=True, desc='Copy ZIP') as bar:
        while chunk := src.read(8 * 1024 * 1024):
            dst.write(chunk)
            bar.update(len(chunk))
    partial.replace(LOCAL_ZIP)

EXTRACT_DIR.mkdir(exist_ok=True)
ready = EXTRACT_DIR / '.ready.json'
reuse = ready.is_file() and json.loads(ready.read_text()).get('manifest_sha256') == EXPECTED_MANIFEST_SHA256
if not reuse:
    with zipfile.ZipFile(LOCAL_ZIP) as archive:
        infos = archive.infolist()
        assert shutil.disk_usage('/content').free > sum(i.file_size for i in infos) + 1_000_000_000, '압축 해제 공간이 부족합니다.'
        seen = set()
        for info in infos:
            relative = PurePosixPath(info.filename.replace('\\', '/'))
            target = (EXTRACT_DIR / str(relative)).resolve()
            if relative.is_absolute() or '..' in relative.parts or not target.is_relative_to(EXTRACT_DIR.resolve()):
                raise ValueError(f'잘못된 ZIP 경로: {info.filename}')
            if stat.S_ISLNK(info.external_attr >> 16) or str(relative) in seen:
                raise ValueError(f'링크 또는 중복 ZIP 경로: {info.filename}')
            seen.add(str(relative))
        for info in tqdm(infos, desc='Extract ZIP'):
            relative = PurePosixPath(info.filename.replace('\\', '/'))
            target = EXTRACT_DIR / str(relative)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as src, target.open('wb') as dst:
                shutil.copyfileobj(src, dst, length=4 * 1024 * 1024)

manifests = list(EXTRACT_DIR.rglob('manifest.jsonl'))
assert len(manifests) == 1, 'manifest.jsonl이 하나인 데이터 ZIP이 필요합니다.'
DATA_DIR = manifests[0].parent
manifest_hash = hashlib.sha256(manifests[0].read_bytes()).hexdigest()
assert manifest_hash == EXPECTED_MANIFEST_SHA256, '이 노트북에 연결된 균등 데이터셋과 ZIP이 다릅니다.'
ready.write_text(json.dumps({'manifest_sha256': manifest_hash}))
print('DATA_DIR:', DATA_DIR)


## 모델 소스
아래 21개 셀은 노트북 생성 시점의 모델·로더·학습·평가·추론·ONNX 내보내기 코드를 로컬에 씁니다. 보통 수정할 필요가 없습니다. 보정 보조 정답은 회전 벡터 3개(라디안)와 위치 차이 3개(월드 단위)이며, 실제 카메라 roll=0°라도 월드 축으로 표현한 회전 보정 벡터의 세 번째 성분이 0일 필요는 없습니다. 정답은 모델 forward 입력이 아닌 보조 loss에만 전달합니다.


In [ ]:
%%writefile /content/gigahands_training/train_hand_transformer.py
"""Train the GigaHands time-based (camera event) pose model: HandTransformer (default) or
the fixed-size, export-friendly HandLite (--arch lite)."""
import argparse
import hashlib
import json
import math
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from hand_tracking.checkpoints import build_model, restore_training, save_checkpoint, training_checkpoint
from hand_tracking.config import ARCHITECTURES, SamplingConfig, add_model_arguments, model_config_from_args
from hand_tracking.data import HandWindows
from hand_tracking.engine import run_epoch

# Settings that must be > 0 and >= 0; the model's own fields are checked by ModelConfig.
POSITIVE = ('epochs', 'batch_size', 'threads', 'lr')
NONNEGATIVE = ('workers', 'seed', 'max_clips', 'max_train_batches', 'max_val_batches', 'bone_weight',
               'weight_decay', 'rig_rotate_deg', 'rig_shift', 'calibration_weight')


def build_parser(argv=None):
    """The model flags depend on --arch, so it is read first."""
    first = argparse.ArgumentParser(add_help=False)
    first.add_argument('--arch', choices=sorted(ARCHITECTURES), default='transformer')
    architecture = first.parse_known_args(argv)[0].arch
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--arch', choices=sorted(ARCHITECTURES), default='transformer',
                   help='transformer: HandTransformer; lite: HandLite (ONNX/ncnn deployment)')
    p.add_argument('--data', default='exports/gigahands_balanced_5gb')
    p.add_argument('--output', default='runs/hand_transformer')
    p.add_argument('--epochs', type=int, default=30)
    p.add_argument('--batch-size', type=int, default=16)
    add_model_arguments(p, ARCHITECTURES[architecture])
    p.add_argument('--max-events', type=int, default=SamplingConfig.max_events,
                   help='Event slots per window (oldest dropped beyond)')
    p.add_argument('--calibration-weight', type=float, default=.01, help='Auxiliary loss on the simulator camera error')
    p.add_argument('--lr', type=float, default=3e-4)
    p.add_argument('--weight-decay', type=float, default=.01)
    p.add_argument('--bone-weight', type=float, default=.1)
    p.add_argument('--rig-rotate-deg', type=float, default=45., help='Train-only random rig+hand rotation; 0 disables')
    p.add_argument('--rig-shift', type=float, default=.25, help='Train-only random translation per axis (world units)')
    p.add_argument('--windows-per-clip', type=int, default=16)
    p.add_argument('--val-windows-per-clip', type=int, default=16)
    p.add_argument('--workers', type=int, default=0)
    p.add_argument('--threads', type=int, default=4)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--device', choices=['auto', 'cpu', 'cuda'], default='auto')
    p.add_argument('--max-clips', type=int, default=0, help='Development subset per split; 0 means all')
    p.add_argument('--max-train-batches', type=int, default=0)
    p.add_argument('--max-val-batches', type=int, default=0)
    p.add_argument('--resume', action='store_true', help='Resume output/last.pt with identical settings')
    return p


def parse_settings(parser, argv=None):
    """Arguments plus validated model and sampling configs; errors exit through the parser."""
    args = parser.parse_args(argv)
    if min(getattr(args, name) for name in POSITIVE) <= 0 or not math.isfinite(args.lr):
        parser.error('Invalid training settings')
    if min(getattr(args, name) for name in NONNEGATIVE) < 0:
        parser.error('Settings must be nonnegative')
    try:
        config = model_config_from_args(args, ARCHITECTURES[args.arch])
        return args, config, SamplingConfig(max_events=args.max_events)
    except ValueError as error:
        parser.error(str(error))


def select_device(name):
    if name == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if name == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable in this PyTorch installation')
    return torch.device(name)


def record_run(parser, args, output):
    """Write run.json, or on resume check it matches; returns the recorded options."""
    options = vars(args).copy()
    options.pop('resume')
    options['manifest_sha256'] = hashlib.sha256((Path(args.data)/'manifest.jsonl').read_bytes()).hexdigest()
    path = output/'run.json'
    if args.resume:
        previous = json.loads(path.read_text())
        # Flags added after that run was started match when they are at their defaults.
        previous.update({k: v for k, v in options.items() if k not in previous and v == parser.get_default(k)})
        if previous != options:
            raise ValueError('Resume settings/dataset differ')
    else:
        path.write_text(json.dumps(options, indent=2), encoding='utf-8')
    return options


def build_loaders(args, config, sampling, device):
    train = HandWindows(args.data, 'train', windows_per_clip=args.windows_per_clip, seed=args.seed,
                        max_clips=args.max_clips, rig_rotate_deg=args.rig_rotate_deg, rig_shift=args.rig_shift,
                        context_s=config.context_s, max_events=sampling.max_events)
    val = HandWindows(args.data, 'val', windows_per_clip=args.val_windows_per_clip, seed=args.seed,
                      max_clips=args.max_clips, context_s=config.context_s, max_events=sampling.max_events)
    return [DataLoader(dataset, batch_size=args.batch_size, num_workers=args.workers,
                       pin_memory=device.type == 'cuda') for dataset in (train, val)]


def main(argv=None):
    parser = build_parser(argv)
    args, config, sampling = parse_settings(parser, argv)
    torch.set_num_threads(args.threads)
    torch.manual_seed(args.seed)
    device = select_device(args.device)
    output = Path(args.output)
    output.mkdir(parents=True, exist_ok=args.resume)
    options = record_run(parser, args, output)
    train_loader, val_loader = build_loaders(args, config, sampling, device)
    model = build_model(args.arch, config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    first, best = 0, float('inf')
    if args.resume:
        first, best = restore_training(output/'last.pt', model, optimizer, scheduler, scaler)
    print(f'device={device} parameters={sum(p.numel() for p in model.parameters()):,} '
          f'train_windows/epoch<={len(train_loader.dataset):,} val_windows<={len(val_loader.dataset):,} '
          f'(events from {config.context_s:.2f} s before the first query)', flush=True)
    if args.max_clips or args.max_train_batches or args.max_val_batches:
        print('DEVELOPMENT RUN: metrics are not full-dataset performance.', flush=True)
    losses = dict(bone_weight=args.bone_weight, calibration_weight=args.calibration_weight)
    for epoch in range(first, args.epochs):
        train_loader.dataset.epoch = epoch
        train = run_epoch(model, train_loader, device, optimizer, scaler, args.max_train_batches, **losses)
        val = run_epoch(model, val_loader, device, limit=args.max_val_batches, **losses)
        scheduler.step()
        improved = val['mpjpe_mm'] < best
        best = min(best, val['mpjpe_mm'])
        with (output/'metrics.jsonl').open('a', encoding='utf-8') as f:
            f.write(json.dumps(dict(epoch=epoch+1, train=train, val=val))+'\n')
        saved = training_checkpoint(model, sampling, optimizer, scheduler, scaler, epoch, best, options)
        save_checkpoint(output/'last.pt', saved)
        if improved:
            save_checkpoint(output/'best.pt', saved)
        print(f"epoch={epoch+1}/{args.epochs} train={train['mpjpe_mm']:.2f}mm val={val['mpjpe_mm']:.2f}mm "
              f"PCK20={val['pck20']:.3f}", flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/evaluate_hand_transformer.py
"""Evaluate a checkpoint, or the model-free triangulation anchor baseline, on fixed validation windows."""
import argparse
from dataclasses import replace
import json
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from hand_tracking.checkpoints import load_checkpoint
from hand_tracking.config import ModelConfig, SamplingConfig
from hand_tracking.data import HandWindows
from hand_tracking.engine import evaluate_anchor, run_epoch


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint', help='Trained model; omit with --triangulation-only')
    p.add_argument('--triangulation-only', action='store_true', help='Evaluate the model-free anchor baseline')
    p.add_argument('--calibration-steps', type=int, default=0,
                   help='Baseline only: per-window Gauss-Newton steps; 0 = nominal')
    p.add_argument('--anchor-lookback-s', type=float, default=ModelConfig.anchor_lookback_s,
                   help='Baseline only; 0 holds the last point')
    p.add_argument('--data', default='exports/gigahands_balanced_5gb')
    p.add_argument('--output', required=True)
    p.add_argument('--batch-size', type=int, default=16)
    p.add_argument('--windows-per-clip', type=int, default=16, help='0 evaluates all validation windows')
    p.add_argument('--workers', type=int, default=0)
    p.add_argument('--device', choices=['cpu', 'cuda'], default='cpu')
    p.add_argument('--max-clips', type=int, default=0, help='Development subset; 0 uses all clips')
    p.add_argument('--max-events', type=int, help='Override checkpoint input layout; baseline default is 128')
    args = p.parse_args(argv)
    if bool(args.checkpoint) == args.triangulation_only:
        p.error('Give exactly one of --checkpoint or --triangulation-only')
    return args


def main(argv=None):
    args = parse_args(argv)
    torch.set_num_threads(4)
    device = torch.device(args.device)
    if args.triangulation_only:
        # Same window/history layout as the default model.
        checkpoint = None
        config = replace(ModelConfig(), anchor_lookback_s=args.anchor_lookback_s)
        sampling = SamplingConfig()
    else:
        checkpoint = load_checkpoint(args.checkpoint, device)
        config, sampling = checkpoint.model.config, checkpoint.sampling
    if args.max_events is not None:
        sampling = SamplingConfig(max_events=args.max_events)
    dataset = HandWindows(args.data, 'val', args.windows_per_clip, max_clips=args.max_clips,
                          context_s=config.context_s, max_events=sampling.max_events)
    loader = DataLoader(dataset, batch_size=args.batch_size, num_workers=args.workers)
    if checkpoint is None:
        report = evaluate_anchor(loader, device, config, args.calibration_steps)
        report.update(method='triangulation_anchor', calibration_steps=args.calibration_steps,
                      anchor_lookback_s=args.anchor_lookback_s)
    else:
        options = checkpoint.saved['training_options']
        report = run_epoch(checkpoint.model, loader, device, bone_weight=options['bone_weight'],
                           calibration_weight=options.get('calibration_weight', 0.))
        report.update(method='transformer', checkpoint=str(Path(args.checkpoint).resolve()))
    report.update(windows_per_clip=args.windows_per_clip, max_events=sampling.max_events,
                  evaluated_clips=len(dataset.rows), limited_clips=bool(args.max_clips))
    with Path(args.output).open('x', encoding='utf-8') as f:
        json.dump(report, f, indent=2)
    print(json.dumps(report), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/predict_hand_transformer.py
"""Predict one NPZ window from a trained checkpoint."""
import argparse
from pathlib import Path
import numpy as np
import torch
from hand_tracking.checkpoints import load_checkpoint
from hand_tracking.contracts import model_inputs
from hand_tracking.data import read_clip, make_sample


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint', required=True)
    p.add_argument('--input', required=True)
    p.add_argument('--output', required=True)
    p.add_argument('--window-index', type=int, default=-1)
    return p.parse_args(argv)


def main(argv=None):
    args = parse_args(argv)
    torch.set_num_threads(4)
    checkpoint = load_checkpoint(args.checkpoint)
    model = checkpoint.model
    clip = read_clip(args.input)
    window = args.window_index % len(clip['window_starts'])
    sample = make_sample(clip, window, context_s=model.config.context_s, max_events=checkpoint.sampling.max_events)
    if not sample['event_present'].any():
        raise ValueError('No arrived observations in this window')
    with torch.inference_mode():
        output = model(*(value[None] for value in model_inputs(sample)), return_details=True)
    latest = int(output.accepted[0].nonzero().max())            # calibration at the newest used event
    pose = output.pose[0, -1].numpy()
    with Path(args.output).open('xb') as stream:
        np.savez_compressed(stream, predicted_xyz=pose, predicted_xyz_cm=pose*clip['world_unit_cm'],
                            target_xyz=sample['target'][-1].numpy(), target_mask=sample['target_mask'][-1].numpy(),
                            world_unit_cm=clip['world_unit_cm'], window_index=window,
                            calibration=output.calibration[0, latest].numpy())
    print(f'Saved {args.output}: [2,21,3] world coordinates and cm')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/export_lite.py
"""Export a trained HandLite checkpoint for deployment (ONNX and/or ncnn) and check the
torch-free LiteRuntime against the PyTorch LiteStream on a real clip.
Requires requirements-export.txt (ncnn: also `ncnn` and `pnnx`)."""
import argparse
import json
from pathlib import Path
import numpy as np
import torch
from hand_tracking.checkpoints import load_checkpoint
from hand_tracking.data import read_clip
from hand_tracking.lite import HandLite, LiteStream
from hand_tracking.lite_export import FORMATS, export_lite
from hand_tracking.lite_runtime import LiteRuntime

BACKENDS = {'onnx': 'onnxruntime', 'ncnn': 'ncnn'}


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint', required=True, help='A checkpoint trained with --arch lite')
    p.add_argument('--output', required=True, help='New directory for the graphs and lite.json')
    p.add_argument('--formats', nargs='+', choices=FORMATS, default=list(FORMATS))
    p.add_argument('--check-input', help='NPZ clip streamed through both runtimes and LiteStream')
    p.add_argument('--queries', type=int, default=30, help='Query times compared from the clip')
    p.add_argument('--tolerance', type=float, default=2e-3,
                   help='Largest accepted |runtime - PyTorch| in world units (ncnn may use fp16)')
    return p.parse_args(argv)


def compare_on_clip(model, directory, formats, npz, queries):
    """Stream the clip's events through LiteStream and each runtime; largest pose difference."""
    clip = read_clip(npz)
    order = np.argsort(clip['event_arrival'], kind='stable')
    times = clip['query_time'][np.linspace(len(clip['query_time'])//4, len(clip['query_time'])-1, queries).astype(int)]
    runners = {'pytorch': LiteStream(model)}
    runners.update({name: LiteRuntime(directory, backend=BACKENDS[name]) for name in formats})
    worst, pushed = {name: 0. for name in formats}, 0
    for time in times:
        while pushed < len(order) and clip['event_arrival'][order[pushed]] <= time:
            i = order[pushed]
            pushed += 1
            for runner in runners.values():
                runner.push(clip['event_camera'][i], clip['event_features'][i], clip['event_valid'][i],
                            clip['event_capture'][i], clip['event_arrival'][i])
        if not pushed:
            continue
        expected = runners['pytorch'].query(time).numpy()
        for name in formats:
            worst[name] = max(worst[name], float(np.abs(runners[name].query(time)-expected).max()))
    return worst


def main(argv=None):
    args = parse_args(argv)
    torch.set_num_threads(4)
    checkpoint = load_checkpoint(args.checkpoint)
    if not isinstance(checkpoint.model, HandLite):
        raise SystemExit('Not a HandLite checkpoint; use export_onnx.py for HandTransformer')
    meta = export_lite(checkpoint.model, args.output, args.formats)
    report = dict(output=str(Path(args.output).resolve()), formats=args.formats, config=meta['config'],
                  graphs=meta['graphs'])
    if args.check_input:
        report['max_abs_difference'] = compare_on_clip(checkpoint.model, args.output, args.formats, args.check_input,
                                                       args.queries)
    print(json.dumps(report, indent=2), flush=True)
    if args.check_input and max(report['max_abs_difference'].values()) > args.tolerance:
        raise SystemExit(f'Runtime differs from PyTorch by more than {args.tolerance}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/export_onnx.py
"""Export a trained checkpoint to ONNX (dynamic batch, event and query counts) and check
it against PyTorch with ONNX Runtime. Requires requirements-export.txt."""
import argparse
import json
from pathlib import Path
import torch
from hand_tracking.checkpoints import load_checkpoint
from hand_tracking.contracts import MODEL_INPUT_KEYS
from hand_tracking.data import read_clip, make_sample
from hand_tracking.export import OUTPUT_NAMES, compare_onnx, example_inputs, export_onnx


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--checkpoint', required=True)
    parser.add_argument('--output', required=True, help='.onnx file to create')
    parser.add_argument('--opset', type=int, help='ONNX opset; default: the exporter default')
    parser.add_argument('--check-input', help='NPZ clip whose last window is also compared (default: synthetic only)')
    parser.add_argument('--tolerance', type=float, default=1e-4, help='Largest accepted |ONNX - PyTorch|')
    return parser.parse_args(argv)


def check_inputs(checkpoint, npz):
    """Synthetic inputs with other sizes than the traced example, plus a real window if given."""
    cases = {'synthetic': example_inputs(batch=3, events=11, queries=4)}
    if npz:
        clip = read_clip(npz)
        sample = make_sample(clip, len(clip['window_starts'])-1, context_s=checkpoint.model.config.context_s,
                             max_events=checkpoint.sampling.max_events)
        cases['npz'] = tuple(sample[key][None] for key in MODEL_INPUT_KEYS)
    return cases


def main(argv=None):
    args = parse_args(argv)
    output = Path(args.output)
    if output.exists():
        raise FileExistsError(output)
    torch.set_num_threads(4)
    checkpoint = load_checkpoint(args.checkpoint)
    export_onnx(checkpoint.model, output, args.opset)
    differences = {name: compare_onnx(output, checkpoint.model, inputs)
                   for name, inputs in check_inputs(checkpoint, args.check_input).items()}
    report = dict(onnx=str(output.resolve()), inputs=list(MODEL_INPUT_KEYS), outputs=list(OUTPUT_NAMES),
                  max_abs_difference=differences, max_events=checkpoint.sampling.max_events,
                  context_s=checkpoint.model.config.context_s)
    print(json.dumps(report, indent=2), flush=True)
    if max(differences.values()) > args.tolerance:
        raise SystemExit(f'ONNX output differs from PyTorch by more than {args.tolerance}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/__init__.py
"""Event-based multi-camera hand tracking components."""


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/checkpoints.py
"""Checkpoints: model and input layout together (including pre-refactor checkpoints), and
the full training state for resuming. `architecture` selects HandTransformer or HandLite;
checkpoints without it are HandTransformer."""
from dataclasses import asdict, dataclass
import torch
from .config import SamplingConfig, config_class
from .lite import HandLite, LiteStream
from .model import HandTransformer
from .stream import EventStream

MODELS = {'transformer': HandTransformer, 'lite': HandLite}
STREAMS = {HandTransformer: EventStream, HandLite: LiteStream}


def architecture_of(model):
    return next(name for name, model_type in MODELS.items() if isinstance(model, model_type))


def build_model(architecture, config):
    return MODELS[architecture](config)


def stream_for(model):
    """Event-by-event runner of either architecture (results equal forward())."""
    return STREAMS[type(model)](model)


INPUT_CONTRACT = ('events [B,E,2,21,14]+valid+camera+capture/arrival seconds+present, query '
                  'times [B,Q] seconds -> pose [B,Q,2,21,3], calibration per event [B,E,3,6] (transformer) '
                  'or per query [B,Q,3,6] (lite); or stream_for(model) event by event; XYZ world units')


@dataclass
class LoadedCheckpoint:
    model: torch.nn.Module
    sampling: SamplingConfig
    saved: dict


def save_checkpoint(path, value):
    """Write atomically: a crash never leaves a truncated checkpoint at path."""
    temporary = path.with_suffix('.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def load_checkpoint(path, device='cpu'):
    saved = torch.load(path, map_location='cpu', weights_only=True)
    architecture = saved.get('architecture', 'transformer')
    model = build_model(architecture, config_class(architecture)(**saved['model_config'])).to(device)
    model.load_state_dict(saved['model'])
    model.eval()
    return LoadedCheckpoint(model, SamplingConfig.from_checkpoint(saved), saved)


def training_checkpoint(model, sampling, optimizer, scheduler, scaler, epoch, best_val_mpjpe_mm, options):
    """Everything needed to resume after `epoch` (0-based) and to load the model alone."""
    cuda = next(model.parameters()).device.type == 'cuda'
    return dict(model=model.state_dict(), architecture=architecture_of(model), model_config=asdict(model.config),
                sampling_config=asdict(sampling),
                optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(), scaler=scaler.state_dict(),
                epoch=epoch, best_val_mpjpe_mm=best_val_mpjpe_mm, rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all() if cuda else None, training_options=options,
                torch_version=str(torch.__version__), parameters=sum(p.numel() for p in model.parameters()),
                input_contract=INPUT_CONTRACT)


def restore_training(path, model, optimizer, scheduler, scaler):
    """Load a training_checkpoint into existing objects; returns (next epoch, best val MPJPE)."""
    saved = torch.load(path, map_location='cpu', weights_only=True)
    model.load_state_dict(saved['model'])
    optimizer.load_state_dict(saved['optimizer'])
    scheduler.load_state_dict(saved['scheduler'])
    scaler.load_state_dict(saved['scaler'])
    torch.set_rng_state(saved['rng'])
    if next(model.parameters()).device.type == 'cuda' and saved['cuda_rng'] is not None:
        torch.cuda.set_rng_state_all(saved['cuda_rng'])
    return saved['epoch']+1, saved['best_val_mpjpe_mm']


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/config.py
"""Model and input-layout settings for both architectures. Torch-free, so deployment
runtimes can read them. The CLI is generated from a config's fields, so a new field is
automatically a training flag (see add_model_arguments)."""
from dataclasses import dataclass, field, fields
from typing import Optional


# Arrival can trail capture by the pipeline delay; windows carry this much extra history.
ARRIVAL_MARGIN_S = .25


def _option(default, help):
    return field(default=default, metadata=dict(help=help))


@dataclass
class ModelConfig:
    dim: int = _option(96, 'Token width')
    heads: int = _option(4, 'Attention heads')
    blocks: int = _option(2, 'Decoder blocks (cross-attention to events, then over joints)')
    encoder_layers: int = _option(1, 'Per-event spatial layers over the 42 joints')
    dropout: float = _option(.1, 'Dropout probability')
    # Scale of the learned per-camera correction (expected rig tolerance) and of the
    # calibration auxiliary loss; defaults match the simulator's error model.
    calibration_rotation_deg: float = _option(5., 'Learned calibration rotation scale (degrees)')
    calibration_position: float = _option(.1, 'Learned calibration shift scale (world units)')
    # Learned calibration at each event from the evidence of events that arrived within
    # calibration_span_s before it (no long-term memory).
    calibration_head: bool = _option(True, 'Learn a per-event camera calibration')
    calibration_span_s: float = _option(.5, 'Evidence window of the learned calibration (s)')
    # A query attends to events captured at most event_span_s before it.
    event_span_s: float = _option(.4, 'Events a query attends to (capture age, s)')
    # A triangulation sample combines each camera's latest ray captured within this age.
    sample_max_age_s: float = _option(.2, 'Oldest ray combined into a triangulation sample (s)')
    # Line fit over samples this recent, extrapolated to the query (0 holds the last sample);
    # samples older than anchor_hold_s are not used.
    anchor_lookback_s: float = _option(.2, 'Anchor velocity fit window (s); 0 holds the last sample')
    anchor_hold_s: float = _option(.3, 'Oldest sample an anchor may hold (s)')
    # Optional (off by default: both lowered anchor accuracy on the simulator data):
    # reject samples with an angular ray residual above this (radians), and fit
    # constant-velocity motion directly to individual rays.
    anchor_ray_gate: Optional[float] = _option(None, 'Reject samples above this angular residual (rad)')
    anchor_motion_fit: bool = _option(False, 'Fit constant-velocity motion to individual rays')

    def __post_init__(self):
        if (self.dim < 1 or self.heads < 1 or self.dim % self.heads or self.blocks < 1
                or self.encoder_layers < 0 or not 0 <= self.dropout < 1
                or min(self.anchor_lookback_s, self.calibration_span_s) < 0
                or min(self.event_span_s, self.sample_max_age_s, self.anchor_hold_s,
                       self.calibration_rotation_deg, self.calibration_position) <= 0
                or (self.anchor_ray_gate is not None and self.anchor_ray_gate <= 0)):
            raise ValueError('Invalid model configuration')

    @property
    def context_s(self):
        """Seconds of events a window needs before its first query so every query gets
        exactly what a stream would give it (anchors, event tokens and their calibration)."""
        calibration = self.calibration_span_s if self.calibration_head else 0.
        return (max(self.event_span_s, self.anchor_hold_s, self.anchor_lookback_s) + ARRIVAL_MARGIN_S
                + calibration + self.sample_max_age_s)


@dataclass
class LiteConfig:
    """HandLite: fixed-size, export-friendly model (hand_tracking.lite)."""
    dim: int = _option(64, 'Token width')
    heads: int = _option(4, 'Attention heads')
    blocks: int = _option(2, 'Decoder blocks (cross-attention to hand tokens, then over joints)')
    dropout: float = _option(.1, 'Dropout probability')
    # A query uses each camera's slots_per_camera most recent events captured within
    # event_span_s: a fixed-size input whatever the frame rate or number of events.
    slots_per_camera: int = _option(8, 'Most recent events per camera a query uses')
    event_span_s: float = _option(.5, 'Oldest event capture a query uses (s)')
    calibration_head: bool = _option(True, 'Learn a per-query camera calibration from the slots')
    calibration_rotation_deg: float = _option(5., 'Learned calibration rotation scale (degrees)')
    calibration_position: float = _option(.1, 'Learned calibration shift scale (world units)')
    sample_max_age_s: float = _option(.2, 'Oldest ray combined into a triangulation sample (s)')
    anchor_lookback_s: float = _option(.2, 'Anchor velocity fit window (s); 0 holds the last sample')
    anchor_hold_s: float = _option(.3, 'Oldest sample an anchor may hold (s)')

    def __post_init__(self):
        if (self.dim < 1 or self.heads < 1 or self.dim % self.heads or self.blocks < 1 or not 0 <= self.dropout < 1
                or self.slots_per_camera < 1 or self.anchor_lookback_s < 0
                or min(self.event_span_s, self.sample_max_age_s, self.anchor_hold_s,
                       self.calibration_rotation_deg, self.calibration_position) <= 0):
            raise ValueError('Invalid model configuration')

    @property
    def context_s(self):
        """Seconds of events a window needs before its first query (slots and their rays)."""
        return max(self.event_span_s, self.anchor_hold_s, self.anchor_lookback_s) + ARRIVAL_MARGIN_S + self.sample_max_age_s


ARCHITECTURES = {'transformer': ModelConfig, 'lite': LiteConfig}


def config_class(architecture):
    try:
        return ARCHITECTURES[architecture]
    except KeyError:
        raise ValueError(f'Unknown architecture {architecture!r}; choose from {sorted(ARCHITECTURES)}') from None


def _flag(name, default):
    """calibration_head (default True) -> --no-calibration-head; others -> --name-with-dashes."""
    if default is True:
        return '--no-'+name.replace('_', '-'), 'no_'+name
    return '--'+name.replace('_', '-'), name


def add_model_arguments(parser, config_type=ModelConfig):
    """One flag per config field, with the field's default and help."""
    group = parser.add_argument_group('model')
    for item in fields(config_type):
        flag, dest = _flag(item.name, item.default)
        if isinstance(item.default, bool):
            text = item.metadata['help']
            group.add_argument(flag, dest=dest, action='store_true', help=f'Disable: {text}' if item.default else text)
        else:
            kind = float if item.default is None else type(item.default)
            group.add_argument(flag, dest=dest, type=kind, default=item.default, help=item.metadata['help'])


def model_config_from_args(args, config_type=ModelConfig):
    values = {}
    for item in fields(config_type):
        _, dest = _flag(item.name, item.default)
        value = getattr(args, dest)
        values[item.name] = (not value) if item.default is True else value
    return config_type(**values)


@dataclass
class SamplingConfig:
    """Input layout settings stored alongside, rather than inside, model weights."""
    max_events: int = 128

    def __post_init__(self):
        if self.max_events < 1:
            raise ValueError('max_events must be positive')

    @classmethod
    def from_checkpoint(cls, saved):
        if 'sampling_config' in saved:
            return cls(**saved['sampling_config'])
        # Checkpoints saved before the module refactor stored this in CLI options.
        options = saved.get('training_options', {})
        return cls(max_events=options.get('max_events', cls.max_events))


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/constants.py
"""Input layout, units and tolerances shared by data, geometry, model and stream."""

NUM_CAMERAS = 3
NUM_HANDS = 2
NUM_JOINTS = 21
# One camera frame gives one token per joint of both hands.
HAND_JOINTS = NUM_HANDS*NUM_JOINTS

# Exported per-joint feature channels [...,FEATURE_CHANNELS]. Channels 8-9 (capture and
# arrival relative to the export query) and 11-13 (ids) are not model inputs: the model
# uses the event's own capture/arrival times instead.
FEATURE_CHANNELS = 14
MODEL_CHANNELS = 11
UV = slice(0, 2)               # normalised pixel coordinates in [0,1]
RAY_ORIGIN = slice(2, 5)       # nominal camera centre, world units
RAY_DIRECTION = slice(5, 8)    # nominal ray direction, world axes
DELAY = slice(10, 11)          # capture->arrival delay, seconds

# Learned features express times, ages and delays in this unit so they are O(1).
TIME_UNIT_S = .1
# Ray-to-point miss vectors are scaled to O(1) for the same reason.
MISS_SCALE = 10.
# Per-camera calibration: rotation vector (3) and centre shift (3).
CALIBRATION_PARAMS = 6

# An event is available to a query when it arrived by the query time, up to float32
# rounding of times relative to the batch origin.
ARRIVAL_TOLERANCE_S = 1e-6
# Distinct sample stamps are >=1/camera_fps/3 apart; float32 relative times differ by ~1e-7.
REPEAT_TOLERANCE_S = 1e-5
# Velocity needs samples spread over at least this time std; extrapolation is capped.
MIN_TIME_STD_S = .005
MAX_EXTRAPOLATION_S = .25
# Triangulation needs two rays at least this far from parallel: sin^2 of the widest
# angle between them; 2e-3 is about 2.6 degrees.
MIN_RAY_SIN2 = 2e-3


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/contracts.py
"""Tensor contracts shared by loaders, model, and streaming inference.

Model times are float32 seconds relative to a nearby origin. Stream storage keeps
absolute timestamps in float64. Shapes below include the batch dimension.
"""
from dataclasses import dataclass
from typing import Callable, NamedTuple, TypedDict
from torch import Tensor
import torch
from .constants import HAND_JOINTS

MODEL_INPUT_KEYS = ('event_features', 'event_valid', 'event_camera', 'event_capture',
                    'event_arrival', 'event_present', 'query_times')


# Supervision and unit conversion; never model inputs.
TARGET_KEYS = ('target', 'target_mask', 'calibration_target', 'world_unit_cm')


def model_inputs(batch):
    return tuple(batch[key] for key in MODEL_INPUT_KEYS)


class WindowSample(TypedDict):
    """One training/evaluation window (data.make_sample); a DataLoader adds a batch dim.
    Times are float32 seconds relative to the window's last query."""
    event_features: Tensor      # [E,2,21,14], invalid joints zeroed
    event_valid: Tensor         # [E,2,21]
    event_camera: Tensor        # [E] long
    event_present: Tensor       # [E], False = padding
    event_capture: Tensor       # [E]
    event_arrival: Tensor       # [E]
    query_times: Tensor         # [Q]
    target: Tensor              # [Q,2,21,3] world units
    target_mask: Tensor         # [Q,2,21]
    calibration_target: Tensor  # [3,6], NaN when unknown (real captures)
    world_unit_cm: Tensor       # []


WINDOW_SAMPLE_KEYS = tuple(WindowSample.__annotations__)
assert set(WINDOW_SAMPLE_KEYS) == set(MODEL_INPUT_KEYS+TARGET_KEYS)


class EventBatch(TypedDict):
    raw: Tensor       # [B,E,2,21,MODEL_CHANNELS], invalid joints zeroed
    valid: Tensor     # [B,E,2,21]
    camera: Tensor    # [B,E]
    capture: Tensor   # [B,E], relative seconds
    arrival: Tensor   # [B,E], relative seconds
    present: Tensor   # [B,E], excludes padding and stale captures


class RawEvents(TypedDict):
    features: Tensor
    valid: Tensor
    camera: Tensor
    capture: Tensor   # stream: [E], absolute float64 seconds
    arrival: Tensor


class ModelOutput(NamedTuple):
    """HandTransformer(..., return_details=True)."""
    pose: Tensor         # [B,Q,2,21,3] world units
    calibration: Tensor  # [B,E,3,6] per-event camera correction (see geometry.apply_calibration)
    accepted: Tensor     # [B,E] events the model used: present and newer than that camera's prior capture


class LayerKV(NamedTuple):
    key: Tensor       # [B,E,42,heads,head_dim]
    value: Tensor


@dataclass
class Evidence:
    summary: Tensor   # [B,E,dim]
    evident: Tensor   # [B,E]

    def map(self, fn: Callable[[Tensor], Tensor]):
        return Evidence(fn(self.summary), fn(self.evident))

    def append(self, other, dim=1):
        return Evidence(torch.cat((self.summary, other.summary), dim),
                        torch.cat((self.evident, other.evident), dim))


@dataclass
class EncodedEvents:
    point: Tensor     # [B,E,2,21,3]
    ok: Tensor        # [B,E,2,21], triangulation validity
    stamp: Tensor     # [B,E,2,21], mean capture time of triangulated rays
    keys: tuple[LayerKV, ...]
    origin: Tensor
    direction: Tensor

    def map(self, fn: Callable[[Tensor], Tensor]):
        """Apply a batch/index/device transform consistently to every cached tensor."""
        return EncodedEvents(fn(self.point), fn(self.ok), fn(self.stamp),
                             tuple(LayerKV(fn(k), fn(v)) for k, v in self.keys),
                             fn(self.origin), fn(self.direction))

    def append(self, other, dim=1):
        def cat(a, b):
            return torch.cat((a, b), dim)
        return EncodedEvents(cat(self.point, other.point), cat(self.ok, other.ok),
                             cat(self.stamp, other.stamp),
                             tuple(LayerKV(cat(a.key, b.key), cat(a.value, b.value))
                                   for a, b in zip(self.keys, other.keys, strict=True)),
                             cat(self.origin, other.origin), cat(self.direction, other.direction))


def to_joint_sequences(value):
    """[B,Q,42,D] -> [B*42,Q,D] for attention along each joint's event history."""
    batch, queries, joints, width = value.shape
    return value.permute(0, 2, 1, 3).reshape(batch*joints, queries, width)


def to_pose_sequences(value, batch):
    """[B*42,Q,D] -> [B,Q,42,D]."""
    return value.reshape(batch, HAND_JOINTS, value.shape[1], value.shape[2]).permute(0, 2, 1, 3)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/data.py
"""Event windows for the time-based model, read from simulator/GigaHands NPZ exports.
Clip-local streaming avoids repeatedly decompressing the same NPZ per window. Samples
follow contracts.WindowSample."""
import json
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import IterableDataset, get_worker_info
from .config import ModelConfig, SamplingConfig
from .constants import NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS, CALIBRATION_PARAMS, RAY_ORIGIN, \
    RAY_DIRECTION
from .contracts import WindowSample

SPLITS = ('train', 'val')
# NPZ arrays a clip keeps: model inputs, targets and window indexing.
CLIP_ARRAYS = ('features', 'input_mask', 'target_xyz', 'target_mask', 'query_time', 'window_starts')
# Frame identity and model-visible (reported) timing, used only to recover events.
TIMING_ARRAYS = ('selected_frame', 'capture_time', 'arrival_time')
CALIBRATION_ARRAYS = ('actual_rotations', 'nominal_rotations', 'actual_origins', 'nominal_origins')


def read_manifest(root):
    """manifest.jsonl rows, checked for valid paths and no participant/source train-val leakage."""
    root = Path(root).resolve()
    lines = (root/'manifest.jsonl').read_text(encoding='utf-8').splitlines()
    rows = [json.loads(line) for line in lines if line.strip()]
    people = {split: set() for split in SPLITS}
    sources = {split: set() for split in SPLITS}
    files = set()
    for row in rows:
        split = row.get('split')
        if split not in people:
            raise ValueError('Manifest requires train/val records without errors')
        path = (root/row['file']).resolve()
        if not path.is_relative_to(root) or not path.is_file():
            raise ValueError('Invalid dataset path')
        if row['file'] in files:
            raise ValueError('Duplicate output path')
        files.add(row['file'])
        people[split].add(row['participant'])
        sources[split].add(row['source'])
    if not all(people.values()):
        raise ValueError('Both train and val must be present')
    if people['train'] & people['val'] or sources['train'] & sources['val']:
        raise ValueError('Participant or source leakage between train and val')
    return rows


def rotation_log(matrix):
    """Rotation matrix -> rotation vector (angles well below 180 degrees)."""
    angle = np.arccos(np.clip((np.trace(matrix)-1)/2, -1, 1))
    axis = np.array([matrix[2,1]-matrix[1,2], matrix[0,2]-matrix[2,0], matrix[1,0]-matrix[0,1]])
    return axis*(angle/(2*np.sin(angle)) if angle > 1e-8 else .5)


def calibration_target(data):
    """Simulator's true correction of the nominal rays [3,6], NaN when unknown (real captures).

    Same convention as geometry.apply_calibration: world-axis rotation about the camera
    centre (R_actual^T R_nominal, since rays are pixel @ R) and centre shift.
    Supervision only; never a model input.
    """
    if not all(key in data for key in CALIBRATION_ARRAYS):
        return np.full((NUM_CAMERAS, CALIBRATION_PARAMS), np.nan, np.float32)
    rows = [np.concatenate([rotation_log(actual.T@nominal), actual_origin-nominal_origin])
            for actual, nominal, actual_origin, nominal_origin in zip(*(data[key] for key in CALIBRATION_ARRAYS))]
    return np.asarray(rows, np.float32)


def read_clip(path):
    with np.load(path, allow_pickle=False) as data:
        clip = {key: data[key].copy() for key in CLIP_ARRAYS}
        clip['calibration_target'] = calibration_target(data)
        timing = {key: data[key].copy() for key in TIMING_ARRAYS}
        meta = json.loads(str(data['metadata']))
    clip['window'] = int(meta['config']['window'])
    clip['world_unit_cm'] = float(meta['units']['world_unit_cm'])
    clip.update(clip_events(clip, timing))
    return clip


def clip_events(clip, timing):
    """Camera frames (events) recovered from the query-grid export, sorted by arrival.

    Each query holds every camera's selected (latest captured, arrived) frame; a frame is
    an event at the first query that selects it, including frames in which no joint was
    detected (they still replace that camera's older rays, as live). Stale out-of-order
    arrivals never get selected, as a stream drops them. A frame superseded before any
    query (two arrivals of one camera between queries) is not recoverable from this format.
    """
    selected = timing['selected_frame']
    new = selected >= 0
    new[1:] &= selected[1:] != selected[:-1]
    frame, camera = np.nonzero(new)
    arrival = timing['arrival_time'][frame, camera].astype(np.float64)
    capture = timing['capture_time'][frame, camera].astype(np.float64)
    # Ties keep (query, camera) order; the model treats input order as arrival order.
    order = np.argsort(arrival, kind='stable')
    return dict(event_features=clip['features'][frame, camera][order],
                event_valid=clip['input_mask'][frame, camera][order],
                event_camera=camera[order].astype(np.int64), event_capture=capture[order],
                event_arrival=arrival[order])


def random_rigid(rng, max_degrees, max_shift):
    """Random rotation (axis uniform, angle up to max_degrees) and translation per axis."""
    axis = rng.normal(size=3)
    axis /= np.linalg.norm(axis)
    angle = np.deg2rad(rng.uniform(-max_degrees, max_degrees))
    k = np.array([[0,-axis[2],axis[1]], [axis[2],0,-axis[0]], [-axis[1],axis[0],0]])
    rotation = np.eye(3)+np.sin(angle)*k+(1-np.cos(angle))*k@k
    return rotation, rng.uniform(-max_shift, max_shift, 3)


def _apply_rigid(features, target, calibration, rigid):
    """Move rig and hands together; pixels are unchanged by a shared rigid transform."""
    rotation, shift = rigid
    features[...,RAY_ORIGIN] = features[...,RAY_ORIGIN]@rotation.T+shift
    features[...,RAY_DIRECTION] = features[...,RAY_DIRECTION]@rotation.T
    # World-axis corrections rotate with the rig; the shift of a difference cancels.
    calibration[:,:3] = calibration[:,:3]@rotation.T
    calibration[:,3:] = calibration[:,3:]@rotation.T
    return target@rotation.T+shift


def make_sample(clip, window_index, rigid=None, context_s=None, max_events=SamplingConfig.max_events) -> WindowSample:
    """Time-based window: the events arrived by the window's last query and captured at most
    context_s before its first query, and the window's query times with their targets.

    Times are seconds relative to the last query. Events are padded to max_events
    (event_present False); if more arrive, the oldest are dropped. rigid=(R, shift)
    moves rig and hands together.
    """
    if context_s is None:
        context_s = ModelConfig().context_s
    start = int(clip['window_starts'][window_index])
    end = start+clip['window']
    query = clip['query_time'][start:end].astype(np.float64)
    last = query[-1]
    arrival, capture = clip['event_arrival'], clip['event_capture']
    chosen = np.nonzero((arrival <= last+1e-9) & (capture >= query[0]-context_s))[0][-max_events:]
    n = len(chosen)
    features = np.zeros((max_events, NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS), np.float32)
    valid = np.zeros((max_events, NUM_HANDS, NUM_JOINTS), bool)
    camera = np.zeros(max_events, np.int64)
    present = np.zeros(max_events, bool)
    event_capture = np.zeros(max_events)
    event_arrival = np.zeros(max_events)
    features[:n] = clip['event_features'][chosen]
    valid[:n] = clip['event_valid'][chosen]
    camera[:n] = clip['event_camera'][chosen]
    present[:n] = True
    event_capture[:n] = capture[chosen]-last
    event_arrival[:n] = arrival[chosen]-last
    target = clip['target_xyz'][start:end].copy()
    calibration = clip['calibration_target'].copy()
    if rigid is not None:
        target = _apply_rigid(features, target, calibration, rigid)
    features[~valid] = 0
    if np.any(event_arrival[present] > 1e-6):
        raise ValueError('Future arrival in historical input')
    as_float = lambda value: torch.from_numpy(value.astype(np.float32))
    return WindowSample(event_features=torch.from_numpy(features), event_valid=torch.from_numpy(valid),
                        event_camera=torch.from_numpy(camera), event_present=torch.from_numpy(present),
                        event_capture=as_float(event_capture), event_arrival=as_float(event_arrival),
                        query_times=as_float(query-last), target=as_float(target),
                        target_mask=torch.from_numpy(clip['target_mask'][start:end].copy()),
                        calibration_target=as_float(calibration),
                        world_unit_cm=torch.tensor(clip['world_unit_cm'], dtype=torch.float32))


class HandWindows(IterableDataset):
    """Windows of one split. Train: shuffled clips, random windows and optional rig
    augmentation, reseeded per epoch (set .epoch). Val: fixed, evenly spaced windows."""

    def __init__(self, root, split, windows_per_clip=16, seed=42, max_clips=0, rig_rotate_deg=0., rig_shift=0.,
                 context_s=None, max_events=SamplingConfig.max_events):
        super().__init__()
        if context_s is None:
            context_s = ModelConfig().context_s
        if (split not in SPLITS or windows_per_clip < 0 or max_clips < 0 or context_s < 0 or max_events < 1
                or min(rig_rotate_deg, rig_shift) < 0):
            raise ValueError('Invalid sampling settings')
        if split == 'val' and (rig_rotate_deg or rig_shift):
            raise ValueError('Rig augmentation is train-only')
        self.root = Path(root)
        self.rows = [row for row in read_manifest(root) if row['split'] == split and row['windows'] > 0]
        if max_clips:
            self.rows = self.rows[:max_clips]
        if not self.rows:
            raise ValueError('No usable clips')
        self.split, self.windows_per_clip, self.seed, self.epoch = split, windows_per_clip, seed, 0
        self.rig_rotate_deg, self.rig_shift = rig_rotate_deg, rig_shift
        self.context_s, self.max_events = context_s, max_events

    def _windows(self, count):
        return min(count, self.windows_per_clip) if self.windows_per_clip else count

    def __len__(self):
        return sum(self._windows(row['windows']) for row in self.rows)

    def __iter__(self):
        training = self.split == 'train'
        indices = np.arange(len(self.rows))
        if training:
            np.random.default_rng(self.seed+self.epoch).shuffle(indices)
        worker = get_worker_info()
        if worker:
            indices = indices[worker.id::worker.num_workers]
        augment = training and (self.rig_rotate_deg or self.rig_shift)
        for index in indices:
            row = self.rows[index]
            clip = read_clip(self.root/row['file'])
            count = len(clip['window_starts'])
            if count != row['windows']:
                raise ValueError('Manifest window count differs from NPZ')
            if training:
                rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, int(index)]))
                chosen = rng.choice(count, self._windows(count), replace=False)
            else:
                chosen = np.linspace(0, count-1, self._windows(count), dtype=int)
            for window in chosen:
                rigid = random_rigid(rng, self.rig_rotate_deg, self.rig_shift) if augment else None
                sample = make_sample(clip, window, rigid, self.context_s, self.max_events)
                # Event-less windows cannot constrain an observation-based estimator;
                # metrics are reported on the final query, so it needs a target.
                if not sample['event_present'].any() or not sample['target_mask'][-1].any():
                    continue
                yield sample


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/engine.py
"""Epoch runners shared by training, validation and the model-free anchor baseline."""
import time
import torch
from .constants import NUM_CAMERAS, CALIBRATION_PARAMS
from .contracts import model_inputs
from .events import event_anchor
from .objectives import PoseTotals, WeightedMean, pose_loss, calibration_loss


def to_device(batch, device):
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}


def batch_loss(model, batch, output, bone_weight, calibration_weight):
    """Pose loss over every query time plus the weighted calibration auxiliary loss.

    Returns (loss, calibration loss, number of known camera-event targets). The simulator's
    true camera correction supervises every accepted event's calibration (never an input);
    padding, dropped captures and real captures (NaN targets) carry none.
    """
    config = model.config
    loss = pose_loss(output.pose, batch['target'], batch['target_mask'], batch['world_unit_cm'], bone_weight)
    target = torch.where(output.accepted[:,:,None,None], batch['calibration_target'][:,None], float('nan'))
    target = target.reshape(-1, NUM_CAMERAS, CALIBRATION_PARAMS)
    auxiliary = calibration_loss(output.calibration.reshape(-1, NUM_CAMERAS, CALIBRATION_PARAMS), target,
                                 config.calibration_rotation_deg, config.calibration_position)
    if calibration_weight and config.calibration_head:
        loss = loss+calibration_weight*auxiliary
    return loss, auxiliary, torch.isfinite(target).all(-1).sum()


def optimizer_step(model, optimizer, scaler, loss):
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)
    scaler.step(optimizer)
    scaler.update()


def run_epoch(model, loader, device, optimizer=None, scaler=None, limit=0, bone_weight=.1, log_every=50,
              calibration_weight=0.):
    """One pass over loader; trains when an optimizer is given. Metrics use the final query."""
    training = optimizer is not None
    model.train(training)
    pose, loss_mean, calibration = PoseTotals(), WeightedMean(), WeightedMean()
    started, batches = time.monotonic(), 0
    for batch in loader:
        batch = to_device(batch, device)
        with torch.set_grad_enabled(training), \
                torch.autocast(device_type=device.type, enabled=training and device.type == 'cuda'):
            output = model(*model_inputs(batch), return_details=True)
            loss, auxiliary, known = batch_loss(model, batch, output, bone_weight, calibration_weight)
        if not torch.isfinite(loss):
            raise FloatingPointError('Nonfinite loss')
        if training:
            optimizer_step(model, optimizer, scaler, loss)
        pose.add(output.pose, batch)
        loss_mean.add(loss.item(), len(output.pose))
        # Match calibration_loss: one item per known camera and event.
        calibration.add(auxiliary.item(), known.item())
        batches += 1
        if batches % log_every == 0:
            print(f"{'train' if training else 'val'} batch={batches} samples={pose.samples} "
                  f"MPJPE={pose.mpjpe_mm:.2f}mm", flush=True)
        if limit and batches >= limit:
            break
    summary = pose.summary()
    return dict(loss=loss_mean.mean, mpjpe_mm=summary['mpjpe_mm'], pck20=summary['pck20'],
                calibration_mse=calibration.mean, samples=summary['samples'], joints=summary['joints'],
                batches=batches, seconds=time.monotonic()-started)


def evaluate_anchor(loader, device, config, calibration_steps=0):
    """Final-query metrics of the model-free anchor with config's anchor settings,
    optionally Gauss-Newton self-calibrated per window."""
    pose = PoseTotals()
    with torch.inference_mode():
        for batch in loader:
            batch = to_device(batch, device)
            anchor = event_anchor(*model_inputs(batch), lookback_s=config.anchor_lookback_s,
                                  hold_s=config.anchor_hold_s, sample_max_age_s=config.sample_max_age_s,
                                  calibration_steps=calibration_steps, gate=config.anchor_ray_gate,
                                  motion_fit=config.anchor_motion_fit, rotation_deg=config.calibration_rotation_deg,
                                  position_std=config.calibration_position)
            pose.add(anchor, batch)
    return pose.summary()


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/events.py
"""Camera events: acceptance, latest-ray selection, triangulation samples and the
query-time anchor. No learned parameters; shared by the model and the baseline."""
import math
import torch
import torch.nn.functional as F
from .config import ModelConfig
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, MODEL_CHANNELS, RAY_ORIGIN,
                        RAY_DIRECTION, MISS_SCALE, TIME_UNIT_S, ARRIVAL_TOLERANCE_S, REPEAT_TOLERANCE_S,
                        MIN_TIME_STD_S, MAX_EXTRAPOLATION_S)
from .contracts import EventBatch, RawEvents
from .geometry import triangulate, apply_calibration, calibrate_cameras, motion_triangulate

# Angular ray noise the Gauss-Newton baseline calibration assumes (radians).
BASELINE_RAY_NOISE = 1.5e-3


def accepted_captures(camera, capture, present):
    """Keep strictly newer captures per camera in arrival order; padding is ignored.

    This is the batch equivalent of EventStream's per-camera high-water mark: an event is
    dropped when an earlier present event of its camera has a capture at least as new.
    Pairwise [B,E,E] instead of a running maximum (cummax has no ONNX operator).
    Comparing the original dtype preserves float64 stream timestamp precision.
    """
    present = present.bool() & (camera >= 0) & (camera < NUM_CAMERAS) & torch.isfinite(capture)
    index = torch.arange(camera.shape[1], device=camera.device)
    earlier = index[None,:,None] > index[None,None,:]                                   # [1,E(i),E(j)]: j before i
    superseded = (earlier & present[:,None,:] & (camera[:,:,None] == camera[:,None,:])
                  & (capture[:,None,:] >= capture[:,:,None])).any(-1)
    return present & ~superseded


def make_events(features, valid, camera, capture, arrival, present) -> EventBatch:
    present = accepted_captures(camera, capture, present)
    return _pack_events(features, valid, camera, capture, arrival, present)


def _pack_events(features, valid, camera, capture, arrival, present) -> EventBatch:
    """Canonical tensor layout after the caller has resolved event acceptance."""
    valid = valid.bool() & present[...,None,None]
    raw = torch.where(valid[...,None], features[...,:MODEL_CHANNELS].float(), 0.)
    return dict(raw=raw, valid=valid, camera=camera.long(), capture=capture.float(), arrival=arrival.float(),
                present=present.bool())


def relative_events(raw: RawEvents, origin: float) -> EventBatch:
    """Convert one stream's absolute float64 timestamps into a relative model batch."""
    present = torch.ones_like(raw['camera'], dtype=torch.bool)[None]
    # Stream buffers contain only accepted frames; do not rescan history per query.
    return _pack_events(raw['features'][None], raw['valid'][None], raw['camera'][None],
                       (raw['capture']-origin)[None], (raw['arrival']-origin)[None], present)


def _gather(values, index):
    """values [B,E,...], index [B,...] -> values at index along the event axis."""
    batch = torch.arange(values.shape[0], device=values.device).reshape(-1, *[1]*(index.ndim-1))
    return values[batch, index]


def _up_to(events, targets):
    """[B,T,E]: events at or before each target in input (arrival) order. Input order,
    not arrival-time comparison, defines causality, so events with equal arrival times
    see each other exactly as when pushed one by one."""
    index = torch.arange(events['camera'].shape[1], device=events['camera'].device)
    return events['present'][:,None] & (index[None,None] <= targets[None,:,None])


def latest_rays(events, targets, max_age):
    """Each camera's most recently captured ray (among events up to the target, captured
    within max_age of its arrival) for every target event: origins/directions
    [B,T,3,2,21,3], mask [B,T,3,2,21], capture [B,T,3]. A late, older capture never
    replaces a newer one. Frames without any detected joint still count as the latest."""
    arrival = events['arrival']; capture = events['capture']
    target_arrival = arrival[:,targets]
    ok = _up_to(events, targets) & (capture[:,None] >= target_arrival[...,None]-max_age)     # [B,T,E]
    scores = torch.stack([torch.where(ok & (events['camera'][:,None] == c), capture[:,None], float('-inf'))
                          for c in range(NUM_CAMERAS)], -1)                              # [B,T,E,3]
    has = torch.isfinite(scores.amax(2)); pick = scores.argmax(2)                        # [B,T,3]
    raw = _gather(events['raw'], pick)                                                   # [B,T,3,2,21,C]
    mask = _gather(events['valid'], pick) & has[...,None,None]
    return raw[...,RAY_ORIGIN], raw[...,RAY_DIRECTION], mask, torch.where(has, _gather(capture, pick), 0.)


def sample_points(origins, directions, mask, capture, params=None, gate=None):
    """Triangulate each target's latest-ray sample, optionally calibrated with that
    target's params [B,T,3,6]. Returns point [B,T,2,21,3], ok [B,T,2,21], stamp (mean
    capture of the used rays) [B,T,2,21] and the calibrated rays."""
    b, t = origins.shape[:2]
    if params is not None:
        origins, directions = apply_calibration(params.reshape(b*t,3,6), origins.reshape(b*t,1,*origins.shape[2:]),
                                                directions.reshape(b*t,1,*directions.shape[2:]))
        origins, directions = origins.reshape(b,t,*origins.shape[2:]), directions.reshape(b,t,*directions.shape[2:])
    origins = torch.where(mask[...,None], origins, 0.); directions = torch.where(mask[...,None], directions, 0.)
    order = (0,1,3,4,2,5)
    point, ok = triangulate(origins.permute(order), directions.permute(order), mask.permute(0,1,3,4,2), gate)
    weight = mask.permute(0,1,3,4,2).float()
    stamp = (capture[:,:,None,None,:]*weight).sum(-1)/weight.sum(-1).clamp_min(1)
    return point, ok, stamp, (origins, directions)


def _own(values, camera):
    """values [B,T,3,...] per camera slot -> the slot of each target's own camera [B,T,...]."""
    index = camera.reshape(*camera.shape, 1, *[1]*(values.ndim-3)).expand(*camera.shape, 1, *values.shape[3:])
    return values.gather(2, index).squeeze(2)


def _miss(point, ok, origin, direction, use):
    """Perpendicular vectors (x MISS_SCALE) from each ray to the point, zero where unusable."""
    relative = point-origin
    miss = relative-direction*(relative*direction).sum(-1, keepdim=True)
    return torch.where((use & ok)[...,None], miss*MISS_SCALE, 0.)


def _gather_joint(values, pick):
    """values [B,E,2,21,C], pick [B,Q,2,21] event index per joint -> [B,Q,2,21,C]."""
    b, e = values.shape[:2]; q = pick.shape[1]; c = values.shape[-1]
    flat = values.reshape(b, e, HAND_JOINTS, c).permute(0,2,1,3)                        # [B,42,E,C]
    index = pick.reshape(b, q, HAND_JOINTS).permute(0,2,1)[...,None].expand(-1,-1,-1,c) # [B,42,Q,C]
    return flat.gather(2, index).permute(0,2,1,3).reshape(b, q, NUM_HANDS, NUM_JOINTS, c)


def query_anchor(point, ok, stamp, arrival, present, query, lookback_s, hold_s, motion=None):
    """Anchor at each query time from the samples of events that arrived by then.

    point/ok/stamp [B,E,2,21,...] per event sample, arrival/present [B,E], query [B,Q].
    A line fitted to the distinct samples captured within lookback_s is extrapolated to
    the query (at most MAX_EXTRAPOLATION_S); otherwise the latest sample within hold_s is
    held. motion = (position, valid, age) from motion_triangulate overrides where valid.
    Joints without a sample use the mean anchor of the same hand, else the origin.
    Returns anchor [B,Q,2,21,3] and flags [B,Q,2,21,5]: latest event's sample valid, joint
    anchor exists, hand anchor exists, age of the latest sample (TIME_UNIT_S), extrapolated.
    """
    e = arrival.shape[1]; index = torch.arange(e, device=arrival.device)
    available = present[:,None] & (arrival[:,None] <= query[...,None]+ARRIVAL_TOLERANCE_S)  # [B,Q,E]
    latest_event = torch.where(available, index, -1).amax(-1)                          # [B,Q]
    now = _gather(ok, latest_event.clamp_min(0)) & (latest_event >= 0)[...,None,None]
    tau = stamp[:,None]-query[:,:,None,None,None]                                      # [B,Q,E,2,21]
    usable = available[...,None,None] & ok[:,None] & (-tau <= hold_s)
    latest = torch.where(usable, index[None,None,:,None,None], -1).amax(2)             # [B,Q,2,21]
    joint = latest >= 0; pick = latest.clamp_min(0)
    anchor = _gather_joint(point, pick)
    age = torch.where(joint, query[...,None,None]-_gather_joint(stamp[...,None], pick)[...,0], 0.)
    moving = torch.zeros_like(joint)
    if lookback_s > 0:
        use = usable & (-tau <= lookback_s)
        # A sample repeated over events (a joint the newest event did not see) counts once.
        same = ((stamp[:,:,None]-stamp[:,None]).abs() <= REPEAT_TOLERANCE_S) & ok[:,:,None] & ok[:,None]
        count = torch.einsum('bsuhj,btuhj->btshj', same.float(), use.float())
        w = torch.where(use, 1/count.clamp_min(1), 0.)
        total = w.sum(2)
        mean_tau = (w*tau).sum(2)/total.clamp_min(1e-12)
        mean_point = (w[...,None]*point[:,None]).sum(2)/total.clamp_min(1e-12)[...,None]
        centred = tau-mean_tau[:,:,None]
        var_tau = (w*centred.square()).sum(2)
        slope = (w[...,None]*centred[...,None]*(point[:,None]-mean_point[:,:,None])).sum(2)
        slope = slope/var_tau.clamp_min(1e-12)[...,None]
        moving = (total >= 2-1e-6) & (var_tau > MIN_TIME_STD_S**2*total)
        anchor = torch.where(moving[...,None], mean_point+slope*(-mean_tau).clamp(0., MAX_EXTRAPOLATION_S)[...,None], anchor)
    if motion is not None:
        position, fitted, fitted_age = motion
        anchor = torch.where(fitted[...,None], position, anchor)
        joint = joint | fitted; moving = moving | fitted; age = torch.where(fitted, fitted_age, age)
    weight = joint[...,None].float()
    hand_mean = (anchor*weight).sum(-2, keepdim=True)/weight.sum(-2, keepdim=True).clamp_min(1)
    hand = joint.any(-1, keepdim=True).expand_as(joint)
    anchor = torch.where(joint[...,None], anchor, hand_mean)
    return anchor, torch.stack((now.float(), joint.float(), hand.float(), age/TIME_UNIT_S, moving.float()), -1)


def _event_motion(events, own_origin, own_direction, query, lookback_s):
    """Motion-fit inputs: each event's own calibrated rays in its camera slot."""
    b, e = events['camera'].shape
    slot = F.one_hot(events['camera'], NUM_CAMERAS).bool()[...,None,None] & events['valid'][:,:,None]   # [B,E,3,2,21]
    spread = lambda v: torch.where(slot[...,None], v[:,:,None], 0.)
    capture = torch.where(slot, events['capture'][:,:,None,None,None], 0.)
    span = events['present'][:,None] & (events['arrival'][:,None] <= query[...,None]+ARRIVAL_TOLERANCE_S)
    return motion_triangulate(spread(own_origin), spread(own_direction), slot, capture, query, span, lookback_s)


def event_anchor(features, valid, camera, capture, arrival, present, query,
                 lookback_s=ModelConfig.anchor_lookback_s, hold_s=ModelConfig.anchor_hold_s,
                 sample_max_age_s=ModelConfig.sample_max_age_s, calibration_steps=0,
                 gate=ModelConfig.anchor_ray_gate, motion_fit=ModelConfig.anchor_motion_fit,
                 rotation_deg=ModelConfig.calibration_rotation_deg, position_std=ModelConfig.calibration_position):
    """Model-free anchor baseline [B,Q,2,21,3] with nominal or per-window Gauss-Newton calibration
    (prior std rotation_deg/position_std, as the learned head's scale)."""
    events = make_events(features, valid, camera, capture, arrival, present)
    targets = torch.arange(camera.shape[1], device=camera.device)
    o, d, mask, cap = latest_rays(events, targets, sample_max_age_s)
    params = None
    if calibration_steps:
        with torch.no_grad():
            window = calibrate_cameras(o, d, mask, calibration_steps, math.radians(rotation_deg), position_std,
                                       BASELINE_RAY_NOISE)
        params = window[:,None].expand(-1, camera.shape[1], -1, -1)
    point, ok, stamp, (co, cd) = sample_points(o, d, mask, cap, params, gate)
    ok = ok & events['present'][...,None,None]
    motion = None
    if motion_fit and lookback_s > 0:
        own = events['camera'][:,targets]
        motion = _event_motion(events, _own(co, own), _own(cd, own), query.float(), lookback_s)
    return query_anchor(point, ok, stamp, events['arrival'], events['present'], query.float(), lookback_s, hold_s, motion)[0]


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/export.py
"""ONNX export of HandTransformer.forward with dynamic batch, event and query counts.

The exported graph takes the seven model inputs (contracts.MODEL_INPUT_KEYS) and returns
pose [B,Q,2,21,3] and per-event calibration [B,E,3,6]. It runs the same code as forward():
the model path avoids operators without an ONNX equivalent (linalg.solve, cross, cummax,
the SDPA decomposition). Needs `pip install -r requirements-export.txt`.

Not exportable: anchor_motion_fit (Cholesky and 6x6 solves). Streaming (EventStream)
stays in PyTorch; the exported graph recomputes every event per call, as forward() does.
"""
from pathlib import Path
import torch
from torch import nn
from .constants import NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS, RAY_ORIGIN, RAY_DIRECTION
from .contracts import MODEL_INPUT_KEYS

OUTPUT_NAMES = ('pose', 'calibration')


class OnnxHandTransformer(nn.Module):
    """forward() with tensor-only outputs and input names matching MODEL_INPUT_KEYS."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, event_features, event_valid, event_camera, event_capture, event_arrival, event_present,
                query_times):
        output = self.model(event_features, event_valid, event_camera, event_capture, event_arrival,
                            event_present, query_times, return_details=True)
        return output.pose, output.calibration


def example_inputs(batch=2, events=6, queries=2):
    """Small valid inputs for tracing. Sizes above 1 keep every axis dynamic in torch.export."""
    features = torch.zeros(batch, events, NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS)
    camera = (torch.arange(events) % NUM_CAMERAS).expand(batch, events).contiguous()
    origins = torch.tensor([[-1., -1., .5], [1., -1., .5], [0., 1., 1.]])
    features[..., RAY_ORIGIN] = origins[camera][:, :, None, None]
    features[..., RAY_DIRECTION] = nn.functional.normalize(-origins[camera], dim=-1)[:, :, None, None]
    capture = (torch.arange(events)*.02-.12).expand(batch, events).contiguous()
    return (features, torch.ones(batch, events, NUM_HANDS, NUM_JOINTS, dtype=torch.bool), camera, capture,
            capture+.03, torch.ones(batch, events, dtype=torch.bool),
            torch.linspace(-.02, 0., queries).expand(batch, queries).contiguous())


def export_onnx(model, path, opset=None):
    """Write model (eval mode) to path as ONNX; returns the path."""
    if model.config.anchor_motion_fit:
        raise ValueError('anchor_motion_fit uses Cholesky and 6x6 solves, which have no ONNX operators')
    model = model.eval()
    batch, events, queries = torch.export.Dim('batch'), torch.export.Dim('events'), torch.export.Dim('queries')
    per_event = {0: batch, 1: events}
    dynamic = dict(zip(MODEL_INPUT_KEYS, [per_event]*6+[{0: batch, 1: queries}]))
    path = Path(path)
    options = {} if opset is None else dict(opset_version=opset)
    with torch.no_grad():
        program = torch.onnx.export(OnnxHandTransformer(model), example_inputs(), dynamo=True,
                                    dynamic_shapes=dynamic, output_names=list(OUTPUT_NAMES), verbose=False,
                                    **options)
    program.save(str(path))
    return path


def compare_onnx(path, model, inputs):
    """Largest |ONNX Runtime - PyTorch| over pose and calibration for these inputs."""
    import onnxruntime
    session = onnxruntime.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    names = [value.name for value in session.get_inputs()]
    if tuple(names) != MODEL_INPUT_KEYS:
        raise ValueError(f'Unexpected ONNX inputs {names}')
    actual = session.run(list(OUTPUT_NAMES), {name: value.numpy() for name, value in zip(names, inputs)})
    with torch.no_grad():
        expected = OnnxHandTransformer(model.eval())(*inputs)
    return max(float(abs(a-e.numpy()).max()) for a, e in zip(actual, expected))


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/geometry.py
"""Differentiable multi-view ray geometry: triangulation, camera corrections and the
optional constant-velocity fit to individual rays. No learned parameters."""
import torch
import torch.nn.functional as F
from .constants import REPEAT_TOLERANCE_S, MIN_TIME_STD_S, MAX_EXTRAPOLATION_S, MIN_RAY_SIN2, TIME_UNIT_S
# Robust motion fit (anchor_motion_fit) residual limits, dimensionless distance/depth.
HUBER_RAY_RESIDUAL = .02
MAX_MOTION_RESIDUAL = .08
MAX_MOTION_RMS = .005


def cross3(a, b):
    """Cross product of [...,3] vectors (torch.linalg.cross has no ONNX operator)."""
    a0, a1, a2 = a.unbind(-1); b0, b1, b2 = b.unbind(-1)
    return torch.stack((a1*b2-a2*b1, a2*b0-a0*b2, a0*b1-a1*b0), -1)


def solve3(a, b):
    """a [...,3,3] invertible, b [...,3] -> a^-1 b by the adjugate (Cramer's rule).

    Exact algebra for 3x3 and exportable (torch.linalg.solve has no ONNX operator);
    callers substitute the identity for singular systems, as for linalg.solve.
    """
    r0, r1, r2 = a.unbind(-2)
    c0, c1, c2 = cross3(r1, r2), cross3(r2, r0), cross3(r0, r1)
    det = (r0*c0).sum(-1, keepdim=True)
    return (b[...,0:1]*c0+b[...,1:2]*c1+b[...,2:3]*c2)/det


def triangulate(origins, directions, mask, max_residual=None):
    """Least-squares point closest to the masked rays.

    origins/directions [...,C,3], mask [...,C] -> point [...,3], valid [...].
    Needs two or more non-parallel rays with positive depths, and with max_residual
    every ray's angular residual within it; otherwise point is 0 and valid is False.
    """
    mask = (mask.bool() & torch.isfinite(origins).all(-1) & torch.isfinite(directions).all(-1)
            & (directions.norm(dim=-1) > 1e-8))
    d = torch.where(mask[...,None], directions, 0.); o = torch.where(mask[...,None], origins, 0.)
    d = d/d.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    eye = torch.eye(3, dtype=d.dtype, device=d.device)
    projection = (eye - d[...,:,None]*d[...,None,:])*mask[...,None,None]
    a = projection.sum(-3)
    rhs = (projection @ o[...,None]).sum(-3)
    # Widest pairwise ray angle instead of eigvalsh: batched CUDA eigh can fail to
    # converge on the empty/single-ray matrices (repeated eigenvalues) present here.
    sin2 = cross3(d[...,:,None,:], d[...,None,:,:]).square().sum(-1).amax((-2,-1))
    valid = (mask.sum(-1) >= 2) & (sin2 > MIN_RAY_SIN2)
    point = solve3(torch.where(valid[...,None,None], a, eye), rhs.squeeze(-1))
    # A point behind any contributing camera means inconsistent rays (outliers), not a hand.
    depth = ((point[...,None,:]-o)*d).sum(-1)
    valid = valid & ((depth > 0) | ~mask).all(-1)
    if max_residual is not None:
        miss = point[...,None,:]-o-depth[...,None]*d
        residual = miss.norm(dim=-1)/depth.clamp_min(1e-6)
        valid = valid & ((residual <= max_residual) | ~mask).all(-1)
    return torch.where(valid[...,None], point, 0.), valid


def _sinc(theta, theta2, small):
    """sin(theta)/theta, Taylor below the `small` threshold (finite value and gradient at 0)."""
    safe = torch.where(small, torch.ones_like(theta), theta)
    return torch.where(small, 1-theta2/6, torch.sin(safe)/safe)


def rodrigues(w):
    """Rotation vectors [...,3] -> matrices [...,3,3]; smooth and differentiable at zero.

    R = I + sinc(t) [w]x + (1-cos t)/t^2 [w]x^2 with (1-cos t)/t^2 = sinc(t/2)^2/2, which
    has no float32 cancellation. No epsilon inside a sqrt: ONNX export folded it away.
    """
    theta2 = w.square().sum(-1)[...,None,None]
    small = theta2 < 1e-8
    theta = torch.where(small, torch.ones_like(theta2), theta2).sqrt()
    a = _sinc(theta, theta2, small)
    b = .5*_sinc(theta/2, theta2/4, small).square()
    zero = torch.zeros_like(w[...,0])
    skew = torch.stack((zero,-w[...,2],w[...,1], w[...,2],zero,-w[...,0], -w[...,1],w[...,0],zero), -1)
    skew = skew.reshape(*w.shape[:-1],3,3)
    eye = torch.eye(3, dtype=w.dtype, device=w.device)
    return eye+a*skew+b*(skew@skew)


def apply_calibration(params, origins, directions):
    """params [B,3,6] (rotation vector, translation per camera); rays [B,T,3,2,21,3].

    Rotates each camera's rays about its centre and shifts the centre, both in world axes.
    """
    rotation = rodrigues(params[...,:3])
    origins = origins+params[:,None,:,None,None,3:]
    directions = torch.einsum('bcxy,btchjy->btchjx', rotation, directions)
    return origins, directions


def _calibration_residuals(params, origins, directions, mask, noise, rotation_std, position_std):
    """Whitened ray-to-point angular residuals plus prior residuals for one window."""
    o, d = apply_calibration(params[None], origins[None], directions[None])
    o, d = o[0].permute(0,2,3,1,4), d[0].permute(0,2,3,1,4)          # [T,2,21,C,3]
    m = mask.permute(0,2,3,1)
    point, ok = triangulate(o, d, m)
    relative = point[...,None,:]-o
    depth = (relative*d).sum(-1, keepdim=True)
    residual = (relative-d*depth)/depth.clamp_min(1e-3)
    residual = torch.where((m & ok[...,None])[...,None], residual, 0.)/noise
    prior = torch.cat(((params[:,:3]/rotation_std).flatten(), (params[:,3:]/position_std).flatten()))
    return torch.cat((residual.flatten(), prior))


def calibrate_cameras(origins, directions, mask, steps, rotation_std, position_std, noise, initial=None):
    """MAP per-camera extrinsic correction from multi-view ray consistency, per window.

    origins/directions [B,T,3,2,21,3], mask [B,T,3,2,21]. Only relative camera errors are
    observable; the Gaussian prior keeps the common (unobservable) rig motion at the
    nominal calibration. Returns params [B,3,6]. Model-free baseline.
    """
    b = origins.shape[0]
    params = origins.new_zeros(b,3,6) if initial is None else initial.to(origins.dtype)
    if steps <= 0: return params
    def residuals(p, o, d, m):
        return _calibration_residuals(p, o, d, m, noise, rotation_std, position_std)
    evaluate = torch.func.vmap(residuals)
    jacobian = torch.func.vmap(torch.func.jacfwd(residuals))
    for _ in range(steps):
        r = evaluate(params, origins, directions, mask)
        j = jacobian(params, origins, directions, mask).reshape(b, r.shape[1], 18)
        # The prior rows make J^T J positive definite, so plain Gauss-Newton is stable.
        # Data rows (1/noise^2) outweigh the prior by ~1e6: accumulate and solve in float64.
        j, r = j.double(), r.double(); jt = j.transpose(1,2)
        step = torch.linalg.solve(jt@j, jt@r[...,None]).squeeze(-1)
        params = params-step.reshape(b,3,6).to(params.dtype)
    return params


def motion_triangulate(origins, directions, valid, capture, query, span, lookback_s):
    """Fit p + v * capture_time directly to individual rays, per query (anchor_motion_fit).

    origins/directions [B,S,C,H,J,3], valid/capture [B,S,C,H,J] (capture on the query's
    time origin), query [B,Q], span [B,Q,S] rays each query may use. Repeated captures
    share one unit of weight; Huber IRLS then rejection/refit handles outliers.
    Underconstrained or inconsistent fits return valid=False. Returns position [B,Q,H,J,3],
    validity and the latest accepted ray's age [B,Q,H,J].
    """
    order = (0,1,3,4,2,5)
    o, d = origins.permute(order), directions.permute(order)  # B,S,H,J,C,3
    m = valid.permute(0,1,3,4,2)
    stamp = capture.permute(0,1,3,4,2)
    finite = torch.isfinite(o).all(-1) & torch.isfinite(d).all(-1) & torch.isfinite(stamp)
    m = m & finite & (d.norm(dim=-1) > 1e-8)
    o, d = torch.where(m[...,None], o, 0.), torch.where(m[...,None], d, 0.)
    d = F.normalize(d, dim=-1)
    stamp = torch.where(m, stamp, 0.)
    age = query[:,:,None,None,None,None]-stamp[:,None]
    use = m[:,None] & span[:,:,:,None,None,None] & (age >= -REPEAT_TOLERANCE_S) & (age <= lookback_s)
    same = ((stamp[:,:,None]-stamp[:,None]).abs() <= REPEAT_TOLERANCE_S) & m[:,:,None] & m[:,None]
    count = torch.einsum('bsuhjc,btuhjc->btshjc', same.float(), use.float())
    base_weight = use.float()/count.clamp_min(1)
    eye3 = torch.eye(3, device=o.device, dtype=o.dtype)
    projection = eye3-d[..., :,None]*d[...,None,:]
    rhs = (projection @ o[...,None]).squeeze(-1)
    # Centre and scale time for the 6x6 system; v below is displacement per TIME_UNIT_S.
    total = base_weight.sum((2,5))
    mean_age = (base_weight*age).sum((2,5))/total.clamp_min(1)
    tau = (mean_age[:,:,None,:,:,None]-age)/TIME_UNIT_S
    eye6 = torch.eye(6, device=o.device, dtype=o.dtype)

    def fit(weight):
        a00 = torch.einsum('btshjc,bshjcxy->bthjxy', weight, projection)
        a01 = torch.einsum('btshjc,bshjcxy->bthjxy', weight*tau, projection)
        a11 = torch.einsum('btshjc,bshjcxy->bthjxy', weight*tau.square(), projection)
        matrix = torch.cat((torch.cat((a00,a01), -1), torch.cat((a01,a11), -1)), -2)
        b0 = torch.einsum('btshjc,bshjcx->bthjx', weight, rhs)
        b1 = torch.einsum('btshjc,bshjcx->bthjx', weight*tau, rhs)
        # Cholesky checks rank without differentiating eigenvectors at repeated
        # eigenvalues. Reject poor pivots before the differentiable solve.
        with torch.no_grad():
            factor, info = torch.linalg.cholesky_ex(matrix.detach())
            pivots = factor.diagonal(dim1=-2, dim2=-1).square()
            cameras = (weight.sum(2) > 0).sum(-1)
            evidence = torch.where(weight > 0, base_weight, 0.).sum((2,5))
            temporal_weight = weight.sum((2,5)).clamp_min(1e-8)
            centre = (weight*tau).sum((2,5))/temporal_weight
            variance = (weight*(tau-centre[:,:,None,:,:,None]).square()).sum((2,5))/temporal_weight
            good = ((info == 0) & (evidence >= 4-1e-5) & (cameras >= 2)
                    & (variance*TIME_UNIT_S**2 > MIN_TIME_STD_S**2)
                    & (pivots.amin(-1) > 1e-4*pivots.amax(-1).clamp_min(1e-8)))
        solution = torch.linalg.solve(torch.where(good[...,None,None], matrix, eye6),
                                      torch.cat((b0,b1), -1)[...,None]).squeeze(-1)
        return solution, good

    def residual(solution):
        position = solution[:,:,None,:,:,None,:3] + tau[...,None]*solution[:,:,None,:,:,None,3:]
        relative = position-o[:,None]
        depth = (relative*d[:,None]).sum(-1)
        angular = (relative-depth[...,None]*d[:,None]).norm(dim=-1)/depth.clamp_min(1e-6)
        return angular, depth

    weight = base_weight
    for _ in range(3):
        solution, good = fit(weight)
        with torch.no_grad():
            angular, depth = residual(solution)
            robust = (HUBER_RAY_RESIDUAL/angular.clamp_min(1e-6)).clamp(max=1)
            weight = base_weight*robust*(depth > 0)
    with torch.no_grad():
        accepted = (angular <= MAX_MOTION_RESIDUAL) & (depth > 0)
        weight = base_weight*accepted
    solution, good = fit(weight)
    with torch.no_grad():
        angular, depth = residual(solution)
        consistent = ((angular <= MAX_MOTION_RESIDUAL) & (depth > 0)) | (weight == 0)
        rms = ((weight*angular.square()).sum((2,5))/weight.sum((2,5)).clamp_min(1)).sqrt()
        # Installation errors or non-linear motion must not be explained as a large
        # spurious velocity. Use the motion model only when its inliers agree well.
        good = good & consistent.all(2).all(-1) & (rms <= MAX_MOTION_RMS) & torch.isfinite(solution).all(-1)
    horizon = mean_age.clamp(min=0, max=MAX_EXTRAPOLATION_S)/TIME_UNIT_S
    point = solution[...,:3]+solution[...,3:]*horizon[...,None]
    latest_age = torch.where(weight > 0, age, float('inf')).amin((2,5)).clamp_min(0)
    return torch.where(good[...,None], point, 0.), good, torch.where(good, latest_age, 0.)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/lite.py
"""HandLite: a fixed-size, export-friendly pose model for asynchronous camera events.

Split so that deployment runs three small fixed-shape networks (ONNX/ncnn) around plain
geometry (lite_runtime, numpy):
  per event, once, on arrival:
    geometry  latest ray of each camera; nominal triangulation; the event's own ray miss
    encoder   21 joint features per hand -> 2 hand tokens                  [2, dim]
  per query at time t:
    slots     each camera's slots_per_camera latest events captured within event_span_s
    calibrator  tokens pooled per camera -> camera correction                 [3, 6]
    geometry  corrected triangulation of every slot's rays; line fit to t -> anchor + flags
    decoder   42 joint queries (anchor) attend to the 2*3*K hand tokens -> offset [42, 3]
The pose is anchor + offset. Networks use only Linear/LayerNorm/GELU/MatMul/Softmax and
additive float masks. forward() computes exactly what LiteStream computes event by event.
"""
from dataclasses import dataclass, replace
import torch
from torch import nn
import torch.nn.functional as F
from .config import LiteConfig
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, FEATURE_CHANNELS, CALIBRATION_PARAMS,
                        UV, RAY_ORIGIN, RAY_DIRECTION, DELAY, TIME_UNIT_S, ARRIVAL_TOLERANCE_S)
from .contracts import ModelOutput
from .events import _gather, _miss, latest_rays, make_events, query_anchor, relative_events, sample_points
from .stream import EventStream, StreamCache

# Per joint: u,v (2), ray origin (3) and direction (3), delay (1), miss vector (3), seen (1),
# triangulated with another camera (1).
JOINT_FEATURES = 14
HAND_FEATURES = NUM_JOINTS*JOINT_FEATURES
# Anchor position (3) and query_anchor flags (5).
ANCHOR_FEATURES = 8
# Additive attention bias of an excluded key; finite so fp16 runtimes stay NaN-free.
MASK_OFF = -1e4


def key_bias(valid):
    """Boolean key validity -> additive attention bias (0 or MASK_OFF)."""
    return (valid.float()-1.)*-MASK_OFF


class Attention(nn.Module):
    """Multi-head attention with an additive bias; 4-D tensors at most (ncnn limit)."""

    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.heads, self.dropout = heads, dropout
        self.query, self.key, self.value, self.out = (nn.Linear(dim, dim) for _ in range(4))

    def forward(self, x, keys, bias=None):
        """x [N,L,d], keys [N,S,d], bias broadcastable to [N,heads,L,S] -> [N,L,d]."""
        n, l, d = x.shape
        s, h = keys.shape[1], self.heads
        q = self.query(x).reshape(n, l, h, d//h).permute(0, 2, 1, 3)
        k = self.key(keys).reshape(n, s, h, d//h).permute(0, 2, 3, 1)
        v = self.value(keys).reshape(n, s, h, d//h).permute(0, 2, 1, 3)
        scores = (q @ k)*(d//h)**-.5
        if bias is not None:
            scores = scores+bias
        weights = F.dropout(scores.softmax(-1), self.dropout, self.training)
        return self.out((weights @ v).permute(0, 2, 1, 3).reshape(n, l, d))


class Block(nn.Module):
    """Pre-norm: optional cross-attention to keys, self-attention, feed-forward."""

    def __init__(self, dim, heads, dropout, cross=True):
        super().__init__()
        self.cross = Attention(dim, heads, dropout) if cross else None
        if cross:
            self.norm_query, self.norm_keys = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.norm_self, self.norm_ff = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attention = Attention(dim, heads, dropout)
        self.feedforward = nn.Sequential(nn.Linear(dim, 2*dim), nn.GELU(), nn.Linear(2*dim, dim))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, keys=None, bias=None):
        if self.cross is not None:
            x = x+self.drop(self.cross(self.norm_query(x), self.norm_keys(keys), bias))
        y = self.norm_self(x)
        x = x+self.drop(self.attention(y, y))
        return x+self.drop(self.feedforward(self.norm_ff(x)))


class EventEncoder(nn.Module):
    """features [N,2,21*14], camera one-hot [N,3] -> hand tokens [N,2,dim]."""

    def __init__(self, dim):
        super().__init__()
        self.joints = nn.Sequential(nn.Linear(HAND_FEATURES, dim), nn.GELU(), nn.Linear(dim, dim))
        self.hand = nn.Parameter(torch.randn(NUM_HANDS, dim)*.02)
        self.camera = nn.Linear(NUM_CAMERAS, dim, bias=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, features, camera):
        return self.norm(self.joints(features)+self.hand+self.camera(camera)[:, None])


class Calibrator(nn.Module):
    """tokens [N,T,dim], pool bias [N,3,T] (camera c's valid tokens 0, others MASK_OFF)
    -> correction [N,3,6] (rotation vector rad, shift world units); zero until trained."""

    def __init__(self, dim, heads, dropout, scale):
        super().__init__()
        self.query = nn.Parameter(torch.randn(NUM_CAMERAS, dim)*.02)
        self.null = nn.Parameter(torch.zeros(1, dim))
        self.norm = nn.LayerNorm(dim)
        self.pool = Attention(dim, heads, dropout)
        self.mix = Block(dim, heads, dropout, cross=False)
        self.output = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, CALIBRATION_PARAMS))
        nn.init.zeros_(self.output[-1].weight)
        nn.init.zeros_(self.output[-1].bias)
        self.register_buffer('scale', scale)

    def forward(self, tokens, pool_bias):
        n, _, d = tokens.shape
        keys = torch.cat((self.null.expand(n, 1, d), self.norm(tokens)), 1)
        bias = torch.cat((torch.zeros_like(pool_bias[:, :, :1]), pool_bias), -1)[:, None]
        pooled = self.pool(self.query.expand(n, NUM_CAMERAS, d), keys, bias)
        return self.output(self.mix(pooled))*self.scale


class Decoder(nn.Module):
    """tokens [N,T,dim], key bias [N,T], capture-to-query gaps [N,T] (TIME_UNIT_S),
    anchor features [N,42,8] -> offset [N,42,3] from the anchor; zero until trained."""

    def __init__(self, dim, heads, blocks, dropout):
        super().__init__()
        self.joints = nn.Parameter(torch.randn(HAND_JOINTS, dim)*.02)
        self.anchor = nn.Sequential(nn.Linear(ANCHOR_FEATURES, dim), nn.GELU(), nn.Linear(dim, dim))
        self.null = nn.Parameter(torch.zeros(1, dim))
        self.time_bias = nn.Sequential(nn.Linear(1, 16), nn.GELU(), nn.Linear(16, heads))
        self.blocks = nn.ModuleList([Block(dim, heads, dropout) for _ in range(blocks)])
        self.output = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, 3))
        nn.init.zeros_(self.output[-1].weight)
        nn.init.zeros_(self.output[-1].bias)

    def forward(self, tokens, token_bias, gaps, anchor_features):
        n, _, d = tokens.shape
        x = self.joints+self.anchor(anchor_features)
        keys = torch.cat((self.null.expand(n, 1, d), tokens), 1)
        bias = self.time_bias(gaps[..., None]).permute(0, 2, 1)+token_bias[:, None]           # [N,h,T]
        bias = torch.cat((torch.zeros_like(bias[..., :1]), bias), -1)[:, :, None]             # [N,h,1,1+T]
        for block in self.blocks:
            x = block(x, keys, bias)
        return self.output(x)


@dataclass
class LiteEncoded:
    """Per-event results that never change after arrival. Times on the batch origin
    (stream storage: absolute float64)."""
    origin: torch.Tensor     # [B,E,3,2,21,3] each camera's latest ray at the event's arrival
    direction: torch.Tensor  # [B,E,3,2,21,3]
    mask: torch.Tensor       # [B,E,3,2,21]
    capture: torch.Tensor    # [B,E,3] capture time of those rays
    evident: torch.Tensor    # [B,E] some own joint triangulated with another camera
    tokens: torch.Tensor     # [B,E,2,dim]

    def map(self, fn):
        return LiteEncoded(*(fn(value) for value in self.__dict__.values()))

    def append(self, other, dim=1):
        return LiteEncoded(*(torch.cat((a, b), dim) for a, b in zip(self.__dict__.values(), other.__dict__.values())))


def joint_features(raw, valid, point, ok):
    """[...,2,21,11] event features, own valid, nominal sample point/ok -> [...,2,21,14]."""
    origin, direction = raw[..., RAY_ORIGIN], raw[..., RAY_DIRECTION]
    miss = _miss(point, ok, origin, direction, valid)
    seen = (ok & valid).float()[..., None]
    x = torch.cat((raw[..., UV]*2-1, origin, direction, raw[..., DELAY]/TIME_UNIT_S, miss, valid.float()[..., None],
                   seen), -1)
    return torch.where(valid[..., None], x, 0.)


class HandLite(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = config or LiteConfig()
        c = self.config
        scale = torch.tensor([c.calibration_rotation_deg*torch.pi/180]*3+[c.calibration_position]*3)
        self.encoder = EventEncoder(c.dim)
        self.calibrator = Calibrator(c.dim, c.heads, c.dropout, scale) if c.calibration_head else None
        self.decoder = Decoder(c.dim, c.heads, c.blocks, c.dropout)

    @property
    def slots(self):
        return NUM_CAMERAS*self.config.slots_per_camera

    def encode_events(self, events, targets=None):
        """LiteEncoded of the target events (default: all), given every earlier event."""
        b, e = events['camera'].shape
        if targets is None:
            targets = torch.arange(e, device=events['camera'].device)
        t = targets.numel()
        with torch.autocast(device_type=events['raw'].device.type, enabled=False):
            origin, direction, mask, capture = latest_rays(events, targets, self.config.sample_max_age_s)
            point, ok, _, _ = sample_points(origin, direction, mask, capture)
            raw, valid = events['raw'][:, targets], events['valid'][:, targets]
            features = joint_features(raw, valid, point, ok)
        camera = F.one_hot(events['camera'][:, targets], NUM_CAMERAS).float()
        tokens = self.encoder(features.reshape(b*t, NUM_HANDS, HAND_FEATURES), camera.reshape(b*t, NUM_CAMERAS))
        evident = (ok & valid).flatten(2).any(-1) & events['present'][:, targets]
        return LiteEncoded(origin, direction, mask, capture, evident, tokens.reshape(b, t, NUM_HANDS, -1))

    def select(self, events, query):
        """Slots [B,Q,3K]: event indices in arrival order (padding first), and validity.
        Each camera's K latest accepted events arrived by the query, captured within span."""
        k = self.config.slots_per_camera
        b, e = events['camera'].shape
        index = torch.arange(e, device=query.device)
        usable = (events['present'][:, None] & (events['arrival'][:, None] <= query[..., None]+ARRIVAL_TOLERANCE_S)
                  & (events['capture'][:, None] >= query[..., None]-self.config.event_span_s))          # [B,Q,E]
        chosen = []
        for camera in range(NUM_CAMERAS):
            score = torch.where(usable & (events['camera'][:, None] == camera), index, -1)
            if e < k:
                score = F.pad(score, (0, k-e), value=-1)
            chosen.append(score.topk(k, dim=-1).values)
        slots = torch.cat(chosen, -1).sort(-1).values
        return slots.clamp_min(0), slots >= 0

    def decode_queries(self, events, encoded, query):
        """Pose [B,Q,2,21,3], calibration [B,Q,3,6] and whether a query had any slot [B,Q]."""
        c = self.config
        b, q = query.shape
        n, s = b*q, self.slots
        slot, valid = self.select(events, query)                                           # [B,Q,S]
        take = lambda value: _gather(value, slot).reshape(n, s, *value.shape[2:])
        camera, capture, arrival = take(events['camera']), take(events['capture']), take(events['arrival'])
        valid = valid.reshape(n, s)
        tokens = take(encoded.tokens).reshape(n, 2*s, -1)
        token_valid = valid.repeat_interleave(NUM_HANDS, -1)
        token_camera = camera.repeat_interleave(NUM_HANDS, -1)
        gaps = ((query.reshape(n, 1)-capture)/TIME_UNIT_S).clamp_min(0.).repeat_interleave(NUM_HANDS, -1)
        cameras = torch.arange(NUM_CAMERAS, device=query.device)[:, None]
        if self.calibrator is not None:
            pool = key_bias((token_camera[:, None] == cameras) & token_valid[:, None])        # [N,3,2S]
            params = self.calibrator(tokens, pool).float()
            # A camera without triangulated evidence in its slots keeps the nominal calibration.
            informative = ((camera[:, None] == cameras) & (valid & take(encoded.evident))[:, None]).any(-1)
            params = torch.where(informative[..., None], params, 0.)
        else:
            params = tokens.new_zeros(n, NUM_CAMERAS, CALIBRATION_PARAMS)
        with torch.autocast(device_type=query.device.type, enabled=False):
            point, ok, stamp, _ = sample_points(take(encoded.origin), take(encoded.direction), take(encoded.mask),
                                                take(encoded.capture), params[:, None].expand(n, s, -1, -1))
            ok = ok & valid[..., None, None]
            anchor, flags = query_anchor(point, ok, stamp, arrival, valid, query.reshape(n, 1),
                                         c.anchor_lookback_s, c.anchor_hold_s)
        features = torch.cat((anchor, flags), -1).reshape(n, HAND_JOINTS, ANCHOR_FEATURES)
        offset = self.decoder(tokens, key_bias(token_valid), gaps, features)
        pose = anchor.reshape(n, HAND_JOINTS, 3)+offset.float()
        return (pose.reshape(b, q, NUM_HANDS, NUM_JOINTS, 3), params.reshape(b, q, NUM_CAMERAS, CALIBRATION_PARAMS),
                valid.any(-1).reshape(b, q))

    def forward(self, event_features, event_valid, event_camera, event_capture, event_arrival, event_present,
                query_times, return_details=False):
        """Same inputs as HandTransformer.forward. return_details gives ModelOutput whose
        calibration is per query [B,Q,3,6] and accepted marks queries with any slot [B,Q]."""
        if event_features.ndim != 5 or event_features.shape[2:] != (NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS):
            raise ValueError('Expected event features [B,E,2,21,14]')
        if event_valid.shape != event_features.shape[:-1] or query_times.ndim != 2:
            raise ValueError('Invalid event/query dimensions')
        events = make_events(event_features, event_valid, event_camera, event_capture, event_arrival, event_present)
        pose, params, rows = self.decode_queries(events, self.encode_events(events), query_times.float())
        return ModelOutput(pose, params, rows) if return_details else pose


class LiteStream(EventStream):
    """HandLite event by event: each event is encoded once (geometry and hand tokens);
    a query runs slot selection, calibrator, corrected anchor and decoder. Equals forward()."""

    def flush(self):
        with torch.inference_mode():
            if not self.pending:
                return
            origin = self.pending[-1].arrival
            raw = self._pending_raw()
            old = self.store
            if old is not None:
                keep = old.raw['capture'] >= origin-self.keep_s
                if not keep.all():
                    old = StreamCache({key: value[keep] for key, value in old.raw.items()},
                                      old.encoded.map(lambda value: value[keep]), None)
                raw = {key: torch.cat((old.raw[key], value)) for key, value in raw.items()}
            count = len(raw['camera'])
            targets = torch.arange(count-len(self.pending), count, device=self.device)
            encoded = self.model.encode_events(relative_events(raw, origin), targets).map(lambda value: value[0])
            encoded = replace(encoded, capture=encoded.capture.double()+origin)
            if old is not None:
                encoded = old.encoded.append(encoded, dim=0)
            self.store = StreamCache(raw, encoded, None)
            self.pending.clear()

    def query(self, time):
        time = float(time)
        latest = self._latest_arrival()
        if latest is None:
            raise ValueError('No events yet')
        if time < latest:
            raise ValueError('Query before the latest arrival')
        self.flush()
        with torch.inference_mode():
            encoded = self.store.encoded.map(lambda value: value[None])
            encoded = replace(encoded, capture=(encoded.capture-time).float())
            query = torch.zeros(1, 1, device=self.device)
            return self.model.decode_queries(relative_events(self.store.raw, time), encoded, query)[0][0, 0]


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/lite_export.py
"""Export HandLite's three fixed-shape networks for lite_runtime: ONNX (onnxruntime) and
ncnn (converted by pnnx). Shapes have no batch axis in ncnn; ONNX graphs take a batch of one.

    encoder     features [2, 21*14], camera one-hot [3]                  -> tokens [2, dim]
    calibrator  tokens [2*3K, dim], pool bias [3, 2*3K]                  -> correction [3, 6]
    decoder     tokens [2*3K, dim], token bias [2*3K], gaps [2*3K], anchor features [42, 8] -> offset [42, 3]
"""
from dataclasses import asdict
import contextlib
import json
import os
from pathlib import Path
import torch
from torch import nn
from .constants import NUM_CAMERAS, NUM_HANDS, HAND_JOINTS
from .lite import ANCHOR_FEATURES, HAND_FEATURES
from .lite_runtime import GRAPHS, META_FILE

FORMATS = ('onnx', 'ncnn')


class _Graph(nn.Module):
    def __init__(self, module):
        super().__init__()
        self.module = module


class EncoderGraph(_Graph):
    def forward(self, features, camera):
        return self.module(features, camera)


class CalibratorGraph(_Graph):
    def forward(self, tokens, pool_bias):
        return self.module(tokens, pool_bias)


class DecoderGraph(_Graph):
    def forward(self, tokens, token_bias, gaps, anchor_features):
        return self.module(tokens, token_bias, gaps, anchor_features)


def graphs(model):
    """name -> (module, example inputs with a batch of one)."""
    model = model.eval()
    dim, tokens = model.config.dim, NUM_HANDS*model.slots
    torch.manual_seed(0)
    token_example = torch.randn(1, tokens, dim)
    result = dict(encoder=(EncoderGraph(model.encoder), (torch.randn(1, NUM_HANDS, HAND_FEATURES),
                                                         torch.eye(NUM_CAMERAS)[:1])),
                  decoder=(DecoderGraph(model.decoder), (token_example, torch.zeros(1, tokens), torch.rand(1, tokens),
                                                         torch.randn(1, HAND_JOINTS, ANCHOR_FEATURES))))
    if model.calibrator is not None:
        result['calibrator'] = (CalibratorGraph(model.calibrator), (token_example, torch.zeros(1, NUM_CAMERAS, tokens)))
    return result


@contextlib.contextmanager
def _working_directory(path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


def export_lite(model, directory, formats=FORMATS):
    """Write the graphs and lite.json (config) to a new directory; returns the metadata."""
    unknown = set(formats)-set(FORMATS)
    if unknown:
        raise ValueError(f'Unknown formats {sorted(unknown)}')
    directory = Path(directory)
    directory.mkdir(parents=True)
    exported = graphs(model)
    if 'calibrator' not in exported:
        # The runtime always loads three graphs; without a head it never calls this one.
        exported['calibrator'] = (CalibratorGraph(_ZeroCalibration()), (torch.zeros(1, 1, model.config.dim),
                                                                          torch.zeros(1, NUM_CAMERAS, 1)))
    with torch.no_grad():
        for name in GRAPHS:
            module, example = exported[name]
            if 'onnx' in formats:
                program = torch.onnx.export(module, example, dynamo=True, verbose=False)
                program.save(str(directory/f'{name}.onnx'))
            if 'ncnn' in formats:
                _pnnx(module, example, directory, name)
    meta = dict(architecture='lite', config=asdict(model.config), formats=list(formats), graphs={
        name: dict(inputs=[list(value.shape[1:]) for value in exported[name][1]]) for name in GRAPHS})
    (directory/META_FILE).write_text(json.dumps(meta, indent=2), encoding='utf-8')
    return meta


class _ZeroCalibration(nn.Module):
    def forward(self, tokens, pool_bias):
        return pool_bias[:, :, :1].expand(-1, -1, 6)*0.


def _pnnx(module, example, directory, name):
    """Convert with pnnx and keep only the ncnn param/bin (pnnx writes next to its input)."""
    import pnnx
    with _working_directory(directory):
        pnnx.export(module, f'{name}.pt', example)
        intermediate = [path for path in Path('.').glob(f'{name}*')
                        if not path.name.endswith(('.ncnn.param', '.ncnn.bin', '.onnx'))
                        or path.name.endswith('.pnnx.onnx')]
        intermediate += list(Path('__pycache__').glob(f'{name}*'))
        for path in intermediate:
            path.unlink()
        cache = Path('__pycache__')
        if cache.is_dir() and not any(cache.iterdir()):
            cache.rmdir()


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/lite_runtime.py
"""HandLite deployment runtime without PyTorch: numpy geometry around the three exported
networks, run by ncnn or ONNX Runtime. Mirrors hand_tracking.lite.LiteStream.

    runtime = LiteRuntime('exports/lite_model', backend='ncnn')
    runtime.push(camera, features [2,21,14], valid [2,21], capture_time, arrival_time)
    pose = runtime.query(time)          # [2,21,3] world units

Times are absolute seconds (float64). Needs numpy plus `ncnn` or `onnxruntime`.
"""
from dataclasses import dataclass
import json
from pathlib import Path
import numpy as np
from .config import LiteConfig
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, MODEL_CHANNELS, UV, RAY_ORIGIN,
                        RAY_DIRECTION, DELAY, TIME_UNIT_S, MISS_SCALE, ARRIVAL_TOLERANCE_S, REPEAT_TOLERANCE_S,
                        MIN_TIME_STD_S, MAX_EXTRAPOLATION_S, MIN_RAY_SIN2)

MASK_OFF = -1e4                 # hand_tracking.lite.MASK_OFF
STREAM_KEEP_MARGIN_S = .5       # hand_tracking.stream.STREAM_KEEP_MARGIN_S
META_FILE = 'lite.json'
GRAPHS = ('encoder', 'calibrator', 'decoder')


# --------------------------------------------------------------------------- geometry

def rodrigues(w):
    """Rotation vectors [...,3] -> matrices [...,3,3]."""
    theta = np.linalg.norm(w, axis=-1)[..., None, None]
    small = theta < 1e-4
    safe = np.where(small, 1., theta)
    a = np.where(small, 1-theta**2/6, np.sin(safe)/safe)
    half = safe/2
    b = np.where(small, .5-theta**2/24, .5*(np.sin(half)/half)**2)
    x, y, z = w[..., 0], w[..., 1], w[..., 2]
    zero = np.zeros_like(x)
    skew = np.stack((zero, -z, y, z, zero, -x, -y, x, zero), -1).reshape(*w.shape[:-1], 3, 3)
    return np.eye(3)+a*skew+b*(skew@skew)


def triangulate(origins, directions, mask):
    """origins/directions [...,C,3], mask [...,C] -> point [...,3], valid [...] (see geometry.triangulate)."""
    mask = (mask & np.isfinite(origins).all(-1) & np.isfinite(directions).all(-1)
            & (np.linalg.norm(directions, axis=-1) > 1e-8))
    d = np.where(mask[..., None], directions, 0.)
    o = np.where(mask[..., None], origins, 0.)
    d = d/np.maximum(np.linalg.norm(d, axis=-1, keepdims=True), 1e-8)
    # sum over rays of (I - d d^T) and (I - d d^T) o; masked rays have d = o = 0.
    count = mask.sum(-1)
    a = count[..., None, None]*np.eye(3)-np.einsum('...ci,...cj->...ij', d, d)
    rhs = (o-d*(d*o).sum(-1, keepdims=True)).sum(-2)
    cameras = d.shape[-2]
    sin2 = np.zeros(count.shape)
    for i in range(cameras):
        for j in range(i+1, cameras):
            sin2 = np.maximum(sin2, np.square(np.cross(d[..., i, :], d[..., j, :])).sum(-1))
    valid = (count >= 2) & (sin2 > MIN_RAY_SIN2)
    a = np.where(valid[..., None, None], a, np.eye(3))
    r0, r1, r2 = a[..., 0, :], a[..., 1, :], a[..., 2, :]
    c0, c1, c2 = np.cross(r1, r2), np.cross(r2, r0), np.cross(r0, r1)
    point = (rhs[..., 0:1]*c0+rhs[..., 1:2]*c1+rhs[..., 2:3]*c2)/(r0*c0).sum(-1, keepdims=True)
    depth = ((point[..., None, :]-o)*d).sum(-1)
    valid = valid & ((depth > 0) | ~mask).all(-1)
    return np.where(valid[..., None], point, 0.), valid


def sample(origins, directions, mask, capture, params=None):
    """Rays [...,3,2,21,3] of the three cameras (optionally corrected by params [3,6]) ->
    point [...,2,21,3], ok [...,2,21], stamp (mean capture of the used rays) [...,2,21]."""
    if params is not None:
        rotation = rodrigues(params[:, :3])                                        # [3,3,3]
        origins = origins+params[:, None, None, 3:]
        directions = np.einsum('cxy,...chjy->...chjx', rotation, directions)
    # [...,camera,hand,joint,xyz] -> [...,hand,joint,camera,xyz]
    rays = list(range(origins.ndim-4))+[origins.ndim-3, origins.ndim-2, origins.ndim-4, origins.ndim-1]
    masks = list(range(mask.ndim-3))+[mask.ndim-2, mask.ndim-1, mask.ndim-3]
    point, ok = triangulate(np.transpose(origins, rays), np.transpose(directions, rays), np.transpose(mask, masks))
    weight = np.transpose(mask, masks).astype(np.float64)
    stamp = (capture[..., None, None, :]*weight).sum(-1)/np.maximum(weight.sum(-1), 1)
    return point, ok, stamp


def anchor_at(point, ok, stamp, query, lookback_s, hold_s):
    """Anchor [2,21,3] and flags [2,21,5] at query from slot samples [S,...] in arrival
    order (all arrived by the query); see events.query_anchor."""
    s = len(point)
    index = np.arange(s)
    now = ok[-1] if s else np.zeros(ok.shape[1:], bool)
    tau = stamp-query
    usable = ok & (-tau <= hold_s)
    latest = np.where(usable, index[:, None, None], -1).max(0)
    joint = latest >= 0
    pick = np.maximum(latest, 0)
    hands, joints = np.indices(joint.shape)
    anchor = point[pick, hands, joints]
    age = np.where(joint, query-stamp[pick, hands, joints], 0.)
    moving = np.zeros_like(joint)
    if lookback_s > 0:
        use = usable & (-tau <= lookback_s)
        same = (np.abs(stamp[:, None]-stamp[None]) <= REPEAT_TOLERANCE_S) & ok[:, None] & ok[None]
        count = (same & use[None]).sum(1)
        w = np.where(use, 1/np.maximum(count, 1), 0.)
        total = w.sum(0)
        mean_tau = (w*tau).sum(0)/np.maximum(total, 1e-12)
        mean_point = (w[..., None]*point).sum(0)/np.maximum(total, 1e-12)[..., None]
        centred = tau-mean_tau
        var_tau = (w*centred**2).sum(0)
        slope = (w[..., None]*centred[..., None]*(point-mean_point)).sum(0)/np.maximum(var_tau, 1e-12)[..., None]
        moving = (total >= 2-1e-6) & (var_tau > MIN_TIME_STD_S**2*total)
        extrapolated = mean_point+slope*np.clip(-mean_tau, 0., MAX_EXTRAPOLATION_S)[..., None]
        anchor = np.where(moving[..., None], extrapolated, anchor)
    weight = joint[..., None].astype(np.float64)
    hand_mean = (anchor*weight).sum(-2, keepdims=True)/np.maximum(weight.sum(-2, keepdims=True), 1)
    hand = np.broadcast_to(joint.any(-1, keepdims=True), joint.shape)
    anchor = np.where(joint[..., None], anchor, hand_mean)
    flags = np.stack((now, joint, hand, age/TIME_UNIT_S, moving), -1).astype(np.float64)
    return anchor, flags


def joint_features(raw, valid, point, ok):
    """Event features [2,21,11], valid, nominal sample -> encoder input [2,21,14]."""
    origin, direction = raw[..., RAY_ORIGIN], raw[..., RAY_DIRECTION]
    relative = point-origin
    miss = relative-direction*(relative*direction).sum(-1, keepdims=True)
    miss = np.where((valid & ok)[..., None], miss*MISS_SCALE, 0.)
    x = np.concatenate((raw[..., UV]*2-1, origin, direction, raw[..., DELAY]/TIME_UNIT_S, miss,
                        valid[..., None], (ok & valid)[..., None]), -1)
    return np.where(valid[..., None], x, 0.)


# --------------------------------------------------------------------------- backends

class OnnxGraphs:
    def __init__(self, directory, threads=1):
        import onnxruntime
        options = onnxruntime.SessionOptions()
        if threads:
            options.intra_op_num_threads = threads
        self.sessions = {name: onnxruntime.InferenceSession(str(Path(directory)/f'{name}.onnx'), options,
                                                            providers=['CPUExecutionProvider'])
                         for name in GRAPHS}

    def run(self, name, *arrays):
        session = self.sessions[name]
        feed = {value.name: np.asarray(array, np.float32)[None] for value, array in zip(session.get_inputs(), arrays)}
        return session.run(None, feed)[0][0]


class NcnnGraphs:
    """ncnn nets. Inputs are copied into ncnn-owned Mats and kept alive until extract():
    the Python binding crashes when an in-place layer writes into numpy-backed memory."""

    def __init__(self, directory, threads=1, fp16=True):
        import ncnn
        self.ncnn = ncnn
        self.nets = {}
        for name in GRAPHS:
            net = ncnn.Net()
            net.opt.use_vulkan_compute = False
            for option in ('use_fp16_packed', 'use_fp16_storage', 'use_fp16_arithmetic'):
                setattr(net.opt, option, fp16)
            if threads:
                net.opt.num_threads = threads
            prefix = str(Path(directory)/name)
            if net.load_param(prefix+'.ncnn.param') or net.load_model(prefix+'.ncnn.bin'):
                raise RuntimeError(f'Cannot load ncnn graph {prefix}')
            self.nets[name] = net

    def run(self, name, *arrays):
        mats = [self.ncnn.Mat(np.ascontiguousarray(array, np.float32)).clone() for array in arrays]
        extractor = self.nets[name].create_extractor()
        for i, mat in enumerate(mats):
            extractor.input(f'in{i}', mat)
        status, output = extractor.extract('out0')
        if status:
            raise RuntimeError(f'ncnn {name} failed with {status}')
        return np.array(output).copy()


# --------------------------------------------------------------------------- runtime

@dataclass
class _Event:
    camera: int
    capture: float
    arrival: float
    origin: np.ndarray     # [3,2,21,3] each camera's latest ray at this arrival
    direction: np.ndarray
    mask: np.ndarray       # [3,2,21]
    ray_capture: np.ndarray  # [3]
    evident: bool
    tokens: np.ndarray     # [2,dim]
    raw: np.ndarray        # [2,21,11], invalid joints zeroed
    valid: np.ndarray      # [2,21]


class LiteRuntime:
    def __init__(self, directory, backend='ncnn', threads=1, fp16=True):
        directory = Path(directory)
        meta = json.loads((directory/META_FILE).read_text(encoding='utf-8'))
        if meta.get('architecture') != 'lite':
            raise ValueError(f'{directory} is not a HandLite export')
        self.config = LiteConfig(**meta['config'])
        if backend == 'ncnn':
            self.graphs = NcnnGraphs(directory, threads, fp16)
        elif backend == 'onnxruntime':
            self.graphs = OnnxGraphs(directory, threads)
        else:
            raise ValueError("backend must be 'ncnn' or 'onnxruntime'")
        self.keep_s = self.config.context_s+STREAM_KEEP_MARGIN_S
        self.reset()

    def reset(self):
        self.events = []
        self.latest = [None]*NUM_CAMERAS          # newest accepted event of each camera

    def __len__(self):
        return len(self.events)

    def push(self, camera, features, valid, capture_time, arrival_time):
        """Encode one arrived camera frame; False (ignored) for a capture no newer than
        that camera's latest. Frames must arrive in order; frames without detections count."""
        camera, capture, arrival = int(camera), float(capture_time), float(arrival_time)
        if self.events and arrival < self.events[-1].arrival:
            raise ValueError('Events must be pushed in arrival order')
        if self.latest[camera] is not None and capture <= self.latest[camera].capture:
            return False
        valid = np.asarray(valid, bool)
        raw = np.where(valid[..., None], np.asarray(features, np.float64)[..., :MODEL_CHANNELS], 0.)
        origin = np.zeros((NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, 3))
        direction = np.zeros_like(origin)
        mask = np.zeros((NUM_CAMERAS, NUM_HANDS, NUM_JOINTS), bool)
        ray_capture = np.zeros(NUM_CAMERAS)
        newest = list(self.latest)
        newest[camera] = _Event(camera, capture, arrival, None, None, None, None, False, None, raw, valid)
        for c, event in enumerate(newest):
            if event is not None and event.capture >= arrival-self.config.sample_max_age_s:
                origin[c], direction[c] = event.raw[..., RAY_ORIGIN], event.raw[..., RAY_DIRECTION]
                mask[c], ray_capture[c] = event.valid, event.capture
        point, ok, _ = sample(origin, direction, mask, ray_capture)
        features = joint_features(raw, valid, point, ok).reshape(NUM_HANDS, -1)
        tokens = self.graphs.run('encoder', features, np.eye(NUM_CAMERAS)[camera])
        event = _Event(camera, capture, arrival, origin, direction, mask, ray_capture, bool((ok & valid).any()),
                       tokens, raw, valid)
        self.latest[camera] = event
        self.events.append(event)
        while self.events and self.events[0].capture < arrival-self.keep_s:
            self.events.pop(0)
        return True

    def slots(self, time):
        """Each camera's K latest events arrived by time and captured within the span, in
        arrival order; None for padding (first)."""
        k, span = self.config.slots_per_camera, self.config.event_span_s
        per_camera = [[] for _ in range(NUM_CAMERAS)]
        for index in range(len(self.events)-1, -1, -1):
            event = self.events[index]
            if event.arrival <= time+ARRIVAL_TOLERANCE_S and event.capture >= time-span \
                    and len(per_camera[event.camera]) < k:
                per_camera[event.camera].append(index)
        chosen = sorted(i for indices in per_camera for i in indices)
        return [None]*(NUM_CAMERAS*k-len(chosen))+[self.events[i] for i in chosen]

    def query(self, time):
        """Pose [2,21,3] at time (not before the latest arrival)."""
        time = float(time)
        if not self.events:
            raise ValueError('No events yet')
        if time < self.events[-1].arrival:
            raise ValueError('Query before the latest arrival')
        pose, _ = self.query_details(time)
        return pose

    def query_details(self, time):
        """Pose [2,21,3] and the camera correction used [3,6]."""
        slots = self.slots(time)
        dim = self.config.dim
        present = np.array([event is not None for event in slots])
        tokens = np.stack([event.tokens if event is not None else np.zeros((NUM_HANDS, dim)) for event in slots])
        tokens = tokens.reshape(-1, dim)
        cameras = np.array([event.camera if event is not None else -1 for event in slots])
        token_valid = np.repeat(present, NUM_HANDS)
        token_camera = np.repeat(cameras, NUM_HANDS)
        params = np.zeros((NUM_CAMERAS, 6))
        if self.config.calibration_head:
            pool = np.where((token_camera[None] == np.arange(NUM_CAMERAS)[:, None]) & token_valid[None], 0., MASK_OFF)
            evident = np.array([event is not None and event.evident for event in slots])
            informative = ((cameras[None] == np.arange(NUM_CAMERAS)[:, None]) & evident[None]).any(-1)
            params = np.where(informative[:, None], self.graphs.run('calibrator', tokens, pool), 0.)
        used = [event for event in slots if event is not None]
        if used:
            point, ok, stamp = sample(np.stack([e.origin for e in used]), np.stack([e.direction for e in used]),
                                      np.stack([e.mask for e in used]), np.stack([e.ray_capture for e in used]), params)
        else:
            point = np.zeros((0, NUM_HANDS, NUM_JOINTS, 3))
            ok = np.zeros((0, NUM_HANDS, NUM_JOINTS), bool)
            stamp = np.zeros((0, NUM_HANDS, NUM_JOINTS))
        anchor, flags = anchor_at(point, ok, stamp, time, self.config.anchor_lookback_s, self.config.anchor_hold_s)
        capture = np.array([event.capture if event is not None else time for event in slots])
        gaps = np.repeat(np.maximum(time-capture, 0.)/TIME_UNIT_S, NUM_HANDS)
        features = np.concatenate((anchor, flags), -1).reshape(HAND_JOINTS, -1)
        offset = self.graphs.run('decoder', tokens, np.where(token_valid, 0., MASK_OFF), gaps, features)
        return anchor+offset.reshape(NUM_HANDS, NUM_JOINTS, 3), params


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/model.py
"""HandTransformer: learned per-event calibration, event tokens and the query decoder
around the model-free anchor of hand_tracking.events."""
import torch
import math
from torch import nn
import torch.nn.functional as F
from .config import ModelConfig
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, FEATURE_CHANNELS, UV, RAY_ORIGIN,
                        RAY_DIRECTION, DELAY, TIME_UNIT_S, CALIBRATION_PARAMS, ARRIVAL_TOLERANCE_S)
from .contracts import EncodedEvents, Evidence, LayerKV, ModelOutput, to_joint_sequences, to_pose_sequences
from .events import make_events, _up_to, latest_rays, sample_points, _own, _miss, query_anchor, _event_motion

# Per-joint ray features: u,v (2), ray origin (3) and direction (3), delay (1), miss vector (3).
RAY_FEATURES = 12
# Anchor position (3) and query_anchor flags (5).
ANCHOR_FEATURES = 8


class CrossAttention(nn.Module):
    """Multi-head attention of queries over keys with a boolean allow-mask and optional
    additive per-head bias (explicit; avoids the NaN-prone fast path of
    nn.TransformerEncoderLayer with per-batch float masks)."""
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.heads = heads; self.dropout = dropout
        self.query = nn.Linear(dim, dim); self.key_value = nn.Linear(dim, 2*dim); self.out = nn.Linear(dim, dim)

    def project_keys(self, keys):
        """keys [...,d] -> key, value [...,h,d/h]; independent of the queries, so cacheable."""
        k, v = self.key_value(keys).reshape(*keys.shape[:-1], 2, self.heads, -1).unbind(-3)
        return k, v

    def attend(self, x, k, v, allowed, bias=None, null=None):
        """x [N,L,d], k/v [N,h,S,d/h], allowed [N,L,S], bias [N,h,L,S].

        null = projected (key, value) [h,d/h] of a learned token that every query may
        attend to with zero bias, so rows without any allowed key stay defined. Without
        it every row of allowed needs one True.
        """
        n, l, d = x.shape; h = self.heads
        if null is not None:
            k, v = (torch.cat((extra[None,:,None].expand(n,-1,-1,-1).to(kv.dtype), kv), 2) for extra, kv in zip(null, (k, v)))
            allowed = torch.cat((allowed.new_ones(n, l, 1), allowed), -1)
            if bias is not None: bias = F.pad(bias, (1, 0))
        q = self.query(x).reshape(n, l, h, d//h).transpose(1,2)
        # Written out rather than F.scaled_dot_product_attention: its export decomposition
        # breaks ONNX conversion, and these sizes (<=43 keys per row) gain nothing from it.
        scores = (q @ k.to(q.dtype).transpose(-1,-2))*(d//h)**-.5
        scores = scores.masked_fill(~allowed[:,None], float('-inf'))
        if bias is not None: scores = scores+bias.to(q.dtype)
        weights = F.dropout(scores.softmax(-1), self.dropout, self.training)
        out = weights @ v.to(q.dtype)
        return self.out(out.transpose(1,2).reshape(n, l, d))

    def forward(self, x, keys, allowed, bias=None, null=None):
        """x [N,L,d], keys [N,S,d], null token [d] or None -> [N,L,d]."""
        k, v = self.project_keys(keys)
        return self.attend(x, k.transpose(1,2), v.transpose(1,2), allowed, bias,
                           None if null is None else self.project_keys(null))


class DecoderLayer(nn.Module):
    """Pre-norm cross-attention of joint queries to event tokens, then FFN."""
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.norm_query = nn.LayerNorm(dim); self.norm_key = nn.LayerNorm(dim); self.norm_ff = nn.LayerNorm(dim)
        self.attention = CrossAttention(dim, heads, dropout)
        self.feedforward = nn.Sequential(nn.Linear(dim, 4*dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(4*dim, dim))
        self.dropout = nn.Dropout(dropout)

    def key_values(self, tokens):
        """Event tokens [...,d] -> this layer's key, value [...,h,d/h] (computed once per event)."""
        return self.attention.project_keys(self.norm_key(tokens))

    def forward(self, x, k, v, allowed, bias, null_token):
        null = self.key_values(null_token)
        x = x+self.dropout(self.attention.attend(self.norm_query(x), k, v, allowed, bias, null))
        return x+self.dropout(self.feedforward(self.norm_ff(x)))


class HandTransformer(nn.Module):
    """Pose at arbitrary query times from asynchronous camera events.

    Each event (one camera frame, 42 joint rays) gets, once, when it arrives:
      1. calibration evidence: its nominal rays against a triangulation with every
         camera's latest ray (frame-local, calibration-independent);
      2. a learned per-camera calibration from the evidence of events that arrived within
         calibration_span_s before it (no long-term memory);
      3. a calibrated triangulation sample and an event token (42 joints, one camera).
    A query at time t anchors each joint on the samples of events arrived by t (line fit
    extrapolated to t), then joint queries attend to the tokens of events captured within
    event_span_s (bias from the capture-to-query gap) and to each other; the output is the
    anchor plus a learned offset. Nothing depends on frame counts or the query rate.

    encode_events() (steps 1-3) and decode_queries() are the two halves of forward();
    EventStream calls them with per-event caching and gets results identical to forward().
    """
    def __init__(self, config=None):
        super().__init__()
        self.config = config or ModelConfig()
        c = self.config
        def mlp(inputs):
            return nn.Sequential(nn.Linear(inputs, c.dim), nn.GELU(), nn.Linear(c.dim, c.dim))
        def layer():
            return nn.TransformerEncoderLayer(c.dim, c.heads, 4*c.dim, c.dropout,
                                               activation='gelu', batch_first=True, norm_first=True)
        self.input_projection = mlp(RAY_FEATURES+CALIBRATION_PARAMS)
        self.anchor_projection = mlp(ANCHOR_FEATURES)
        self.camera_embedding = nn.Embedding(NUM_CAMERAS, c.dim)
        self.hand_embedding = nn.Embedding(NUM_HANDS, c.dim)
        self.joint_embedding = nn.Embedding(NUM_JOINTS, c.dim)
        self.query_embedding = nn.Parameter(torch.zeros(c.dim))
        self.event_null = nn.Parameter(torch.zeros(c.dim))
        # Per-event spatial encoder over the 42 joints seen by one camera.
        self.encoder = nn.ModuleList([layer() for _ in range(c.encoder_layers)])
        # Attention bias per head from the capture-to-query gap (seconds, no frame index).
        self.time_bias = nn.Sequential(nn.Linear(1, 32), nn.GELU(), nn.Linear(32, c.heads))
        self.cross = nn.ModuleList([DecoderLayer(c.dim, c.heads, c.dropout) for _ in range(c.blocks)])
        self.spatial = nn.ModuleList([layer() for _ in range(c.blocks)])
        self.output = nn.Sequential(nn.LayerNorm(c.dim), nn.Linear(c.dim, c.dim), nn.GELU(), nn.Linear(c.dim, 3))
        # Zero residual heads: an untrained model reproduces the uncalibrated anchor.
        nn.init.zeros_(self.output[-1].weight); nn.init.zeros_(self.output[-1].bias)
        if c.calibration_head:
            self.evidence_projection = mlp(RAY_FEATURES)
            self.calibration_query = nn.Parameter(torch.zeros(NUM_CAMERAS, c.dim))
            self.calibration_null = nn.Parameter(torch.zeros(c.dim))
            self.calibration_pool = CrossAttention(c.dim, c.heads, c.dropout)
            self.calibration_mix = layer()
            self.calibration_output = nn.Sequential(nn.LayerNorm(c.dim), nn.Linear(c.dim, c.dim), nn.GELU(),
                                                    nn.Linear(c.dim, CALIBRATION_PARAMS))
            nn.init.zeros_(self.calibration_output[-1].weight); nn.init.zeros_(self.calibration_output[-1].bias)
        self.register_buffer('calibration_scale', torch.tensor(
            [math.radians(c.calibration_rotation_deg)]*3+[c.calibration_position]*3))

    def _joints(self, device):
        return (self.hand_embedding(torch.arange(NUM_HANDS, device=device))[:,None] +
                self.joint_embedding(torch.arange(NUM_JOINTS, device=device))[None]).reshape(HAND_JOINTS, self.config.dim)

    @staticmethod
    def _ray_features(raw, origin, direction, miss):
        """u,v in [-1,1], ray origin/direction, capture->arrival delay (TIME_UNIT_S) and miss vector."""
        return torch.cat((raw[...,UV]*2-1, origin, direction, raw[...,DELAY]/TIME_UNIT_S, miss), -1)

    def _evidence(self, events, targets, rays):
        """Per-event calibration evidence summary [B,T,d] and whether it has any [B,T]."""
        c = self.config; b = events['camera'].shape[0]; t = targets.numel(); d = c.dim
        with torch.no_grad(), torch.autocast(device_type=events['raw'].device.type, enabled=False):
            point, ok, _, _ = sample_points(*rays)
            raw = events['raw'][:,targets]; own = events['valid'][:,targets]
            origin, direction = raw[...,RAY_ORIGIN], raw[...,RAY_DIRECTION]
            miss = _miss(point, ok, origin, direction, own)
            x = self._ray_features(raw, origin, direction, miss)
        use = own & ok
        x = self.evidence_projection(torch.where(own[...,None], x, 0.)).reshape(b, t, HAND_JOINTS, d)
        x = x+self.camera_embedding(events['camera'][:,targets])[:,:,None]+self._joints(x.device)
        weight = use.reshape(b, t, HAND_JOINTS, 1).float()
        return Evidence((x*weight).sum(2)/weight.sum(2).clamp_min(1), use.flatten(2).any(-1))

    def _calibrate(self, events, evidence, targets):
        """Calibration [B,T,3,6] at each target event from evidence that arrived within span."""
        c = self.config; b, e = events['camera'].shape; t = targets.numel(); d = c.dim; h = c.heads
        arrival = events['arrival']; target_arrival = arrival[:,targets]
        cameras = torch.arange(NUM_CAMERAS, device=arrival.device)
        band = _up_to(events, targets) & (arrival[:,None] > target_arrival[...,None]-c.calibration_span_s)   # [B,T,E]
        allowed = band[:,:,None] & (events['camera'][:,None,None] == cameras[:,None])     # [B,T,3,E]
        # Every camera's query pools over the same evidence keys: project them once.
        k, v = (value.transpose(1,2)[:,None].expand(b, NUM_CAMERAS, h, e, d//h).reshape(b*NUM_CAMERAS, h, e, d//h)
                for value in self.calibration_pool.project_keys(evidence.summary))
        mask = allowed.permute(0,2,1,3).reshape(b*NUM_CAMERAS, t, e)
        query = (self.calibration_query+self.camera_embedding.weight)[None,:,None].expand(b,NUM_CAMERAS,t,d)
        pooled = self.calibration_pool.attend(query.reshape(b*NUM_CAMERAS,t,d), k, v, mask,
                                              null=self.calibration_pool.project_keys(self.calibration_null))
        pooled = pooled.reshape(b,NUM_CAMERAS,t,d).permute(0,2,1,3).reshape(b*t,NUM_CAMERAS,d)
        correction = self.calibration_output(self.calibration_mix(pooled)).float().reshape(b,t,NUM_CAMERAS,CALIBRATION_PARAMS)
        # A camera with no evidence (fresh ray seen with another camera) keeps the nominal calibration.
        informative = (allowed & evidence.evident[:,None,None]).any(-1)
        return torch.where(informative[...,None], correction*self.calibration_scale, 0.)

    def _encode(self, events, params, targets, rays):
        """Calibrated sample (point, ok, stamp), own calibrated rays and event tokens [B,T,42,d]."""
        c = self.config; b = events['camera'].shape[0]; t = targets.numel(); d = c.dim
        camera = events['camera'][:,targets]
        with torch.autocast(device_type=events['raw'].device.type, enabled=False):
            point, ok, stamp, (co, cd) = sample_points(*rays, params.float(), c.anchor_ray_gate)
            raw = events['raw'][:,targets]; own = events['valid'][:,targets]
            own_origin, own_direction = _own(co, camera), _own(cd, camera)
            miss = _miss(point, ok, own_origin, own_direction, own)
            correction = (_own(params, camera)/self.calibration_scale)[:,:,None,None]
            correction = correction.expand(b, t, NUM_HANDS, NUM_JOINTS, CALIBRATION_PARAMS)
            x = torch.cat((self._ray_features(raw, own_origin, own_direction, miss), correction), -1)
        x = self.input_projection(torch.where(own[...,None], x, 0.)).reshape(b, t, HAND_JOINTS, d)
        x = x+self.camera_embedding(camera)[:,:,None]+self._joints(x.device)
        x = x.reshape(b*t, HAND_JOINTS, d)
        for layer in self.encoder: x = layer(x)
        ok = ok & events['present'][:,targets,None,None]
        tokens = x.reshape(b, t, HAND_JOINTS, d)
        # Decoder keys/values depend only on the event: cache them with it.
        keys = tuple(LayerKV(*layer.key_values(tokens)) for layer in self.cross)
        return EncodedEvents(point, ok, stamp, keys, own_origin, own_direction)

    def encode_events(self, events, targets=None, previous_evidence=None):
        """Encode target events (default: all) given every earlier event in `events`.

        Returns (EncodedEvents, calibration [B,T,3,6], evidence of the targets or None).
        previous_evidence covers the events before the targets (a stream's cached history);
        without it the targets are the whole history. Nothing is stored on the module.
        """
        if targets is None:
            targets = torch.arange(events['camera'].shape[1], device=events['camera'].device)
        with torch.autocast(device_type=events['raw'].device.type, enabled=False):
            rays = latest_rays(events, targets, self.config.sample_max_age_s)
        evidence = None
        if self.config.calibration_head:
            evidence = self._evidence(events, targets, rays)
            history = evidence if previous_evidence is None else previous_evidence.append(evidence)
            params = self._calibrate(events, history, targets)
        else:
            params = events['raw'].new_zeros(events['camera'].shape[0], targets.numel(), NUM_CAMERAS, CALIBRATION_PARAMS)
        return self._encode(events, params, targets, rays), params, evidence

    def decode_queries(self, events, encoded, query):
        """Pose [B,Q,2,21,3] at query times [B,Q] (same time origin as the events)."""
        c = self.config; b, e = events['camera'].shape; q = query.shape[1]; d = c.dim; h = c.heads
        with torch.autocast(device_type=query.device.type, enabled=False):
            motion = None
            if c.anchor_motion_fit and c.anchor_lookback_s > 0:
                motion = _event_motion(events, encoded.origin, encoded.direction, query, c.anchor_lookback_s)
            anchor, flags = query_anchor(encoded.point, encoded.ok, encoded.stamp, events['arrival'],
                                         events['present'], query, c.anchor_lookback_s, c.anchor_hold_s, motion)
        x = torch.cat((anchor, flags), -1).reshape(b, q, HAND_JOINTS, ANCHOR_FEATURES)
        x = self._joints(query.device)+self.anchor_projection(x)+self.query_embedding
        gap = query[:,:,None]-events['capture'][:,None]                                      # [B,Q,E]
        allowed = (events['present'][:,None] & (events['arrival'][:,None] <= query[...,None]+ARRIVAL_TOLERANCE_S)
                   & (gap <= c.event_span_s))
        allowed = allowed[...,None] & events['valid'].reshape(b, 1, e, HAND_JOINTS)          # [B,Q,E,42]
        allowed = allowed.permute(0,3,1,2).reshape(b*HAND_JOINTS, q, e)
        bias = self.time_bias(gap.clamp_min(0)[...,None]/TIME_UNIT_S).permute(0,3,1,2)      # [B,h,Q,E]
        bias = bias[:,None].expand(b, HAND_JOINTS, h, q, e).reshape(b*HAND_JOINTS, h, q, e)
        x = to_joint_sequences(x)
        for cross, spatial, kv in zip(self.cross, self.spatial, encoded.keys, strict=True):
            k, v = (value.permute(0,2,3,1,4).reshape(b*HAND_JOINTS, h, e, d//h) for value in kv)
            x = cross(x, k, v, allowed, bias, self.event_null)
            x = spatial(to_pose_sequences(x, b).reshape(b*q, HAND_JOINTS, d))
            x = to_joint_sequences(x.reshape(b, q, HAND_JOINTS, d))
        x = to_pose_sequences(x, b)
        return anchor+self.output(x).reshape(b, q, NUM_HANDS, NUM_JOINTS, 3).float()

    def forward(self, event_features, event_valid, event_camera, event_capture, event_arrival, event_present,
                query_times, return_details=False):
        """Events [B,E,2,21,14] (u,v · ray origin/direction · times · delay · ids), valid
        [B,E,2,21], camera [B,E], capture/arrival [B,E] seconds, present [B,E], sorted by
        arrival (ties: input order is arrival order); query_times [B,Q] seconds on the same
        origin -> pose [B,Q,2,21,3].

        Padding and captures no newer than that camera's prior maximum are ignored.
        A query uses only events that arrived by its time. return_details gives a
        ModelOutput that adds each event's calibration [B,E,3,6] (rotation vector rad,
        translation world units; see apply_calibration) and which events were accepted.
        No actual calibration or ground truth is used as input.
        """
        if event_features.ndim != 5 or event_features.shape[2:] != (NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS):
            raise ValueError('Expected event features [B,E,2,21,14]')
        if event_valid.shape != event_features.shape[:-1] or query_times.ndim != 2:
            raise ValueError('Invalid event/query dimensions')
        events = make_events(event_features, event_valid, event_camera, event_capture, event_arrival, event_present)
        encoded, params, _ = self.encode_events(events)
        pose = self.decode_queries(events, encoded, query_times.float())
        return ModelOutput(pose, params, events['present']) if return_details else pose


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/objectives.py
"""Training losses (pose, bone length, calibration auxiliary) and evaluation totals."""
from dataclasses import dataclass
import math
import torch
import torch.nn.functional as F


def calibration_loss(prediction, target, rotation_deg=5., position_std=.1):
    """Mean squared calibration error in prior-std units over cameras with a known target.

    prediction/target [N,3,6]; target rows of NaN (e.g. real captures, padding) are ignored.
    The common rig motion is unobservable, so the optimum is its posterior mean.
    """
    scale = prediction.new_tensor([math.radians(rotation_deg)]*3+[position_std]*3)
    known = torch.isfinite(target).all(-1)
    error = ((prediction-torch.where(known[...,None], target, 0.))/scale).square().mean(-1)
    return (error*known).sum()/known.sum().clamp_min(1)


BONES = [(0,1+4*f) for f in range(5)] + [(1+4*f+j,2+4*f+j) for f in range(5) for j in range(3)]


def pose_loss(prediction, target, target_mask, world_unit_cm, bone_weight=.1, beta_m=.006):
    """Masked SmoothL1 plus target-specific bone length loss, in meters.

    prediction/target [B,...,2,21,3]; world_unit_cm [B] converts each sample, so
    beta and the loss scale do not depend on the export's world unit.
    """
    valid = target_mask.bool()
    scale = (world_unit_cm/100).reshape(-1, *[1]*(prediction.ndim-1))
    prediction = prediction.float()*scale
    safe_target = torch.where(valid[...,None], target, 0.)*scale
    joints = F.smooth_l1_loss(prediction, safe_target, beta=beta_m, reduction='none').mean(-1)
    position = (joints*valid).sum()/valid.sum().clamp_min(1)
    a, b = map(list, zip(*BONES))
    pred_lengths = (prediction[...,a,:]-prediction[...,b,:]).norm(dim=-1)
    true_lengths = (safe_target[...,a,:]-safe_target[...,b,:]).norm(dim=-1)
    bone_mask = valid[...,a] & valid[...,b]
    bone = ((pred_lengths-true_lengths).abs()*bone_mask).sum()/bone_mask.sum().clamp_min(1)
    return position+bone_weight*bone


def error_totals(prediction, target, mask, world_unit_cm):
    """MPJPE numerator (mm), PCK@20mm numerator and valid joint count for [B,2,21,3]."""
    valid = mask.bool()
    distance = (prediction-torch.where(valid[...,None],target,0.)).norm(dim=-1)
    mm = distance*world_unit_cm[:,None,None]*10
    return (mm*valid).sum(), ((mm<20)&valid).sum(), valid.sum()


@dataclass
class PoseTotals:
    """Running MPJPE and PCK@20mm over the final query of each window (the reported metric)."""
    error_mm: float = 0.
    correct: int = 0
    joints: int = 0
    samples: int = 0

    def add(self, pose, batch):
        """pose [B,Q,2,21,3] against a batch's target/target_mask/world_unit_cm."""
        with torch.no_grad():
            error, correct, joints = error_totals(pose[:,-1].float(), batch['target'][:,-1],
                                                  batch['target_mask'][:,-1], batch['world_unit_cm'])
        self.error_mm += error.item()
        self.correct += correct.item()
        self.joints += joints.item()
        self.samples += len(pose)

    @property
    def mpjpe_mm(self):
        return self.error_mm/self.joints if self.joints else float('nan')

    def summary(self):
        if not self.joints:
            raise ValueError('No valid observations/targets in this loader')
        return dict(mpjpe_mm=self.mpjpe_mm, pck20=self.correct/self.joints, samples=self.samples, joints=self.joints)


@dataclass
class WeightedMean:
    total: float = 0.
    weight: float = 0.

    def add(self, value, weight):
        self.total += value*weight
        self.weight += weight

    @property
    def mean(self):
        """None when nothing was weighted (e.g. no known calibration target)."""
        return self.total/self.weight if self.weight else None


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/stream.py
"""Arrival-ordered event buffering and inference caches, separate from model math."""
from dataclasses import dataclass, replace
from typing import NamedTuple
import torch
from .contracts import RawEvents, EncodedEvents, Evidence
from .events import relative_events

STREAM_KEEP_MARGIN_S = .5


class PendingEvent(NamedTuple):
    features: torch.Tensor
    valid: torch.Tensor
    camera: int
    capture: float
    arrival: float


@dataclass
class StreamCache:
    """Unbatched tensors with absolute float64 capture, arrival and sample times."""
    raw: RawEvents
    encoded: EncodedEvents
    evidence: Evidence | None

    def select(self, keep):
        return StreamCache({key: value[keep] for key, value in self.raw.items()},
                           self.encoded.map(lambda value: value[keep]),
                           None if self.evidence is None else self.evidence.map(lambda value: value[keep]))


class EventStream:
    """Cache each accepted camera event once; query arbitrary current/future times.

    Events must arrive in order. Empty detections replace older camera rays;
    duplicate or older captures are dropped, matching the batch event policy.
    push() owns its input tensors. query()/flush() batch all pending encodings.
    """

    def __init__(self, model):
        self.model = model.eval()
        self.config = model.config
        self.device = next(model.parameters()).device
        # Pruning keeps a margin beyond the exact context a new event can reach.
        self.keep_s = self.config.context_s + STREAM_KEEP_MARGIN_S
        self.reset()

    def reset(self):
        self.store: StreamCache | None = None
        self.pending: list[PendingEvent] = []
        self.newest_capture: dict[int, float] = {}

    def __len__(self):
        stored = 0 if self.store is None else len(self.store.raw['camera'])
        return stored + len(self.pending)

    def _latest_arrival(self):
        if self.pending:
            return self.pending[-1].arrival
        return None if self.store is None else float(self.store.raw['arrival'][-1])

    def push(self, camera, features, valid, capture_time, arrival_time):
        """Own a frame; return False when its capture does not advance that camera."""
        arrival_time = float(arrival_time)
        capture_time = float(capture_time)
        camera = int(camera)
        latest = self._latest_arrival()
        if latest is not None and arrival_time < latest:
            raise ValueError('Events must be pushed in arrival order')
        if capture_time <= self.newest_capture.get(camera, float('-inf')):
            return False
        event = PendingEvent(torch.as_tensor(features, dtype=torch.float32, device=self.device).clone(),
                             torch.as_tensor(valid, dtype=torch.bool, device=self.device).clone(),
                             camera, capture_time, arrival_time)
        self.newest_capture[camera] = capture_time
        self.pending.append(event)
        return True

    def _pending_raw(self):
        return RawEvents(
            features=torch.stack([event.features for event in self.pending]),
            valid=torch.stack([event.valid for event in self.pending]),
            camera=torch.tensor([event.camera for event in self.pending], device=self.device),
            capture=torch.tensor([event.capture for event in self.pending], dtype=torch.float64, device=self.device),
            arrival=torch.tensor([event.arrival for event in self.pending], dtype=torch.float64, device=self.device))

    @torch.inference_mode()
    def flush(self):
        """Encode pending events transactionally: a failure leaves the buffer intact."""
        if not self.pending:
            return
        origin = self.pending[-1].arrival
        raw = self._pending_raw()
        old = self.store
        if old is not None:
            keep = old.raw['capture'] >= origin-self.keep_s
            if not keep.all():
                old = old.select(keep)
            raw = {key: torch.cat((old.raw[key], value)) for key, value in raw.items()}
        count = len(raw['camera'])
        targets = torch.arange(count-len(self.pending), count, device=self.device)
        previous = None if old is None or old.evidence is None else old.evidence.map(lambda value: value[None])
        encoded, _, evidence = self.model.encode_events(relative_events(raw, origin), targets, previous)
        encoded = encoded.map(lambda value: value[0])
        encoded = replace(encoded, stamp=encoded.stamp.double()+origin)
        evidence = None if evidence is None else evidence.map(lambda value: value[0])
        if old is not None:
            encoded = old.encoded.append(encoded, dim=0)
            if evidence is not None:
                evidence = old.evidence.append(evidence, dim=0)
        self.store = StreamCache(raw, encoded, evidence)
        self.pending.clear()

    @torch.inference_mode()
    def query(self, time):
        time = float(time)
        latest = self._latest_arrival()
        if latest is None:
            raise ValueError('No events yet')
        if time < latest:
            raise ValueError('Query before the latest arrival')
        self.flush()
        stored = self.store
        encoded = stored.encoded.map(lambda value: value[None])
        encoded = replace(encoded, stamp=(encoded.stamp-time).float())
        query = torch.zeros(1, 1, device=self.device)
        return self.model.decode_queries(relative_events(stored.raw, time), encoded, query)[0,0]


In [ ]:
# 5. 데이터 분리·파일 크기와 소스 확인
import importlib
importlib.invalidate_caches()
from hand_tracking.checkpoints import build_model
from hand_tracking.config import config_class
from hand_tracking.data import read_manifest
SOURCE_HASHES = {'train_hand_transformer.py': '41b5e97f55730456d0d2ba98466d20b2b44061fe292b8a6b32589cf206674f1e', 'evaluate_hand_transformer.py': '349fa9034a582fd396a648c7fd5ff0ccba9d713a7f6afbe8322a8c68a0a0d82b', 'predict_hand_transformer.py': 'd39b7fb0410a90dbdbfd567f253e9517ae30c180fe04ce1f219b61f8f1d24922', 'export_lite.py': 'ae277bae7a7e43500d2cad38358e0e5217281a829718c5bb289995e3e19d1618', 'export_onnx.py': '023a3f50a9ede8bc5e6b14f439c0837a0e7dbfb9452ea5aa3a8fe04cd967aa70', 'hand_tracking/__init__.py': '166e0ecdd3de289e24bb3e1b27966969d9e37e6ae1e6599069c69d4ce2b7c07e', 'hand_tracking/checkpoints.py': 'e4ff4c99ac01426fb2e8f6c549dfcafc69f6039bc812449480d6cc25a633cb2b', 'hand_tracking/config.py': '9b766662240ac1920231e48f9fcaabc437e3da6ba3e74d1de3378e69260fc8f3', 'hand_tracking/constants.py': '6084115e94a5f5b051f754a2489e9cee8d3d59e850a2972355a1e1d7b13689ce', 'hand_tracking/contracts.py': '523ccb49c930e8062d440bb2fcc5cad342887d052c1cc18ed5891c428a289256', 'hand_tracking/data.py': '58420e91a4684b0e43f321ab3ffcba00fd6fbadc25366ec93434e5c2d035edc3', 'hand_tracking/engine.py': 'dd4f20fb14866e108dd12d29d0b615499bf2a56fd7e3cb2d057dbdd806feaae9', 'hand_tracking/events.py': 'd008102783fc336c000649905d3c188f2e8417be8a17532d68cf6af529a5efe7', 'hand_tracking/export.py': 'b8c96bcd8e26154be6e0826c59f07e0fdf8c3dc96b97262b65609eacbba788f0', 'hand_tracking/geometry.py': '727a3c7251686c237385e66454113e482d030ff104c32cad7c96e868d56f9adb', 'hand_tracking/lite.py': '408847c983c2563c230e3ad31c00722bbd34e31bfed8a595cd323aa85944fdee', 'hand_tracking/lite_export.py': '28aab265fdc91b20325935de29a9221464aa47583ad5c9d93138f290c9bb93cb', 'hand_tracking/lite_runtime.py': 'c7f4fa9e776479d460b21db4d263b28c1bddb7d7a8f9ae3c0b37508b154f28ce', 'hand_tracking/model.py': '3349b709eaf2e4bb1748fd6ed1260ec54d74248e4d04a6074d51418d12c4280c', 'hand_tracking/objectives.py': '25ffd37f0111b3e1943bec5caf847d8f29dd58b02200ce8ae8fc54b200505048', 'hand_tracking/stream.py': '0060ce954c9dc7da5b59aa44682e9639a5f854d42e9af4e8c635f3502eae5420'}
for name, expected in SOURCE_HASHES.items():
    actual = hashlib.sha256((WORK_DIR / name).read_text().encode()).hexdigest()
    assert actual == expected, f'소스 변경 감지: {name}. 변경 실험은 노트북과 실행 이름을 새로 만드세요.'
rows = read_manifest(DATA_DIR)  # 참가자/원본 train-val 중복이면 오류
for row in tqdm(rows, desc='Check dataset'):
    assert (DATA_DIR / row['file']).stat().st_size == row['bytes'], row['file']
for split in ('train', 'val'):
    selected = [r for r in rows if r['split'] == split]
    print(split, 'clips:', len(selected), 'participants:', len({r['participant'] for r in selected}))
cfg = config_class(ARCH)(dropout=DROPOUT, **{key[2:].replace('-', '_'): value for key, value in MODEL_OPTIONS.items()})
temporary_model = build_model(ARCH, cfg)
print('parameters:', f'{sum(p.numel() for p in temporary_model.parameters()):,}')
del temporary_model

def run_python(script, *arguments):
    command = [sys.executable, '-u', str(WORK_DIR / script), *map(str, arguments)]
    print(' '.join(command), flush=True)
    process = subprocess.Popen(command, cwd=WORK_DIR, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise


## GPU 동작 확인
본학습 전에 실제 데이터의 작은 배치로 CUDA·mixed precision·역전파·체크포인트 저장을 확인합니다. 이 결과는 성능 평가가 아니며 본학습 가중치로 사용하지 않습니다. GPU 메모리가 부족하면 배치 크기를 줄이고 다시 실행하세요.


In [ ]:
# 6. GPU smoke test — 본학습 실행 폴더와 분리
import uuid
smoke_dir = WORK_DIR / ('smoke_' + uuid.uuid4().hex[:8])
run_python('train_hand_transformer.py',
           '--data', DATA_DIR, '--output', smoke_dir, '--epochs', 1,
           '--batch-size', min(BATCH_SIZE, 2), *MODEL_ARGS, '--calibration-weight', CALIBRATION_WEIGHT,
           '--max-clips', 4, '--max-train-batches', 2, '--max-val-batches', 2,
           '--windows-per-clip', 2, '--val-windows-per-clip', 2,
           '--workers', 0, '--device', 'cuda')
print('GPU smoke test passed.')


## 본학습 / 재개
- 기본값은 학습 클립마다 epoch당 최대 16개 무작위 윈도우입니다. 윈도우는 query 시각 16개와, 첫 query보다 약 1.35초 전부터 마지막 query까지 도착한 이벤트(최대 128개)입니다.
- 검증 구간은 고정됩니다. MPJPE는 마지막 query의 절대 3D 관절 오차(mm)이며 작을수록 좋습니다.
- 카메라 보정은 최근 약 0.5초의 이벤트에서만 추정합니다(장기 기억 없음; lite는 카메라당 최근 이벤트 8개).
- 카메라/손 전체에 같은 강체 변환을 적용하는 학습 증강이 활성화됩니다. 이것은 데이터 생성기의 개별 카메라 설치 오차와 별개입니다.
- 보조 정답 `calibration_target[3,6]`은 카메라마다 회전 보정 벡터(rad) 3개와 위치 보정(world) 3개입니다. 모델 입력에는 넣지 않고 `CALIBRATION_WEIGHT`로 보조 손실을 조절합니다. 회전 벡터의 세 번째 성분은 카메라의 roll 각도와 다르므로 roll 0°에서도 0이 아닐 수 있습니다.
- **매 epoch 완료 시** Drive에 `last.pt`, 개선 시 `best.pt`를 저장합니다. 런타임이 끊기면 완료되지 않은 epoch는 다시 학습합니다.
- anchor는 이벤트마다 카메라별 최신 ray를 삼각측량한 표본에 직선을 맞춰 query 시각으로 외삽합니다. ray 궤적 직접 풀이와 잔차 게이트는 시뮬레이션 데이터에서 정확도를 낮춰 기본으로 끕니다. 이전 구조의 가중치는 쓸 수 없으니 새 실행으로 학습하세요.
- 같은 설정·같은 `RUN_NAME`으로 다시 실행하면 마지막 완료 epoch부터 재개합니다. 재개 중 배치 크기·epoch 수·모델 설정을 바꾸면 거부됩니다. 설정 변경은 새 `RUN_NAME`으로 시작하세요.
- 첫 epoch가 끝나기 전에 중단되어 `last.pt`가 없으면 새 `RUN_NAME`을 사용하세요. 기존 결과를 자동 삭제하지 않습니다.


In [ ]:
# 7. Train / Resume — Drive에 epoch마다 저장
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
source_file = EXPERIMENT_DIR / 'source_hashes.json'
if source_file.is_file():
    assert json.loads(source_file.read_text()) == SOURCE_HASHES, '이 실행의 모델 소스가 달라졌습니다. 새 RUN_NAME을 사용하세요.'
else:
    source_file.write_text(json.dumps(SOURCE_HASHES, indent=2))
    snapshot = EXPERIMENT_DIR / 'source_snapshot'
    snapshot.mkdir(exist_ok=True)
    for name in SOURCE_HASHES:
        (snapshot / name).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(WORK_DIR / name, snapshot / name)

resume = (RUN_DIR / 'last.pt').is_file()
if RUN_DIR.exists() and not resume:
    raise RuntimeError('완료된 체크포인트가 없는 기존 실행 폴더입니다. RUN_NAME을 새 이름으로 바꾸고 설정 셀부터 다시 실행하세요.')
arguments = [
    '--data', DATA_DIR, '--output', RUN_DIR, '--epochs', EPOCHS,
    '--batch-size', BATCH_SIZE, *MODEL_ARGS, '--lr', LEARNING_RATE, '--calibration-weight', CALIBRATION_WEIGHT,
    '--windows-per-clip', WINDOWS_PER_CLIP, '--val-windows-per-clip', VAL_WINDOWS_PER_CLIP,
    '--rig-rotate-deg', RIG_ROTATE_DEG, '--rig-shift', RIG_SHIFT_WORLD,
    '--workers', WORKERS, '--threads', 2, '--seed', SEED, '--device', 'cuda',
]
if resume:
    arguments.append('--resume')
    print('Resume:', RUN_DIR / 'last.pt')
run_python('train_hand_transformer.py', *arguments)
print('체크포인트:', RUN_DIR / 'best.pt')


In [ ]:
# 8. 학습 곡선
import matplotlib.pyplot as plt
records = [json.loads(line) for line in (RUN_DIR / 'metrics.jsonl').read_text().splitlines() if line.strip()]
# 중단 시 마지막 완료 epoch의 기록이 반복될 수 있어 epoch별 마지막 행 표시
records = list({r['epoch']: r for r in records}.values())
records.sort(key=lambda r: r['epoch'])
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for split in ('train', 'val'):
    axes[0].plot([r['epoch'] for r in records], [r[split]['mpjpe_mm'] for r in records], label=split)
    axes[1].plot([r['epoch'] for r in records], [r[split]['pck20'] * 100 for r in records], label=split)
    axes[2].plot([r['epoch'] for r in records], [r[split]['calibration_mse'] for r in records], label=split)
axes[0].set(title='MPJPE', xlabel='Epoch', ylabel='mm')
axes[1].set(title='PCK @ 20 mm', xlabel='Epoch', ylabel='%')
axes[2].set(title='Calibration auxiliary loss', xlabel='Epoch', ylabel='Normalized MSE')
for ax in axes:
    ax.grid(alpha=.3)
    ax.legend()
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR / 'learning_curves.png', dpi=150)
plt.show()


## 검증: 학습 모델 vs 삼각측량 기준선
동일한 검증 윈도우에서 비교합니다. 기본은 모든 검증 클립의 최대 4개 윈도우입니다. `EVAL_WINDOWS_PER_CLIP=0`으로 바꾸면 전체 검증 윈도우를 평가하므로 시간이 더 걸립니다. 기준선은 학습 없이 nominal 캘리브레이션으로 만든 삼각측량 anchor입니다. 검증은 모델 선택에 사용되었으므로 독립된 최종 test 성능은 아닙니다.


In [ ]:
# 9. 같은 조건의 모델/기준선 평가
EVAL_WINDOWS_PER_CLIP = 4
assert (RUN_DIR / 'best.pt').is_file(), '먼저 학습 체크포인트를 생성하세요.'
eval_dir = EXPERIMENT_DIR / ('eval_' + uuid.uuid4().hex[:8])
eval_dir.mkdir()
common = ['--data', DATA_DIR, '--windows-per-clip', EVAL_WINDOWS_PER_CLIP,
          '--batch-size', BATCH_SIZE, '--workers', WORKERS, '--device', 'cuda']
run_python('evaluate_hand_transformer.py', '--triangulation-only', '--output', eval_dir / 'anchor.json',
           '--anchor-lookback-s', MODEL_OPTIONS['--anchor-lookback-s'], *common)
run_python('evaluate_hand_transformer.py', '--checkpoint', RUN_DIR / 'best.pt', '--output', eval_dir / 'model.json', *common)
anchor_report = json.loads((eval_dir / 'anchor.json').read_text())
model_report = json.loads((eval_dir / 'model.json').read_text())
assert anchor_report['samples'] == model_report['samples']
print(f"Anchor MPJPE: {anchor_report['mpjpe_mm']:.2f} mm")
print(f"Model  MPJPE: {model_report['mpjpe_mm']:.2f} mm")
print(f"Reduction: {anchor_report['mpjpe_mm'] - model_report['mpjpe_mm']:.2f} mm (양수이면 개선)")
print('결과 저장:', eval_dir)


In [ ]:
# 10. 검증 클립 1개 예측 및 3D 비교
chosen = next(row for row in rows if row['split'] == 'val' and row['windows'] > 0)
prediction_file = EXPERIMENT_DIR / ('prediction_' + uuid.uuid4().hex[:8] + '.npz')
run_python('predict_hand_transformer.py', '--checkpoint', RUN_DIR / 'best.pt',
           '--input', DATA_DIR / chosen['file'], '--output', prediction_file)
with np.load(prediction_file, allow_pickle=False) as data:
    predicted = data['predicted_xyz_cm'].copy()
    target = data['target_xyz'] * float(data['world_unit_cm'])
    valid = data['target_mask'].copy()
from hand_tracking.objectives import BONES
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')
for hand, color in enumerate(('tab:blue', 'tab:orange')):
    for a, b in BONES:
        if valid[hand, a] and valid[hand, b]:
            ax.plot(*target[hand, [a,b]].T, color=color, linestyle='--', alpha=.6)
        ax.plot(*predicted[hand, [a,b]].T, color=color)
ax.set(xlabel='X (cm)', ylabel='Y (cm)', zlabel='Z (cm)', title='Prediction: solid / Ground truth: dashed')
combined = np.concatenate((predicted.reshape(-1,3), target[valid]), axis=0)
center = (combined.min(0)+combined.max(0))/2
radius = max(float(np.ptp(combined,axis=0).max())/2, 1.)
ax.set_xlim(center[0]-radius,center[0]+radius)
ax.set_ylim(center[1]-radius,center[1]+radius)
ax.set_zlim(center[2]-radius,center[2]+radius)
ax.set_box_aspect((1,1,1))
plt.show()
print('원본:', chosen['source'])
print('저장:', prediction_file)


## 배포용 내보내기 (선택)
`ARCH='lite'`이면 `export_lite.py`가 신경망 3개(이벤트 encoder·보정기·decoder)를 **ncnn과 ONNX**로 내보내고, 라즈베리 파이용 런타임(`hand_tracking/lite_runtime.py`: numpy 기하 + ncnn)과 PyTorch 스트림의 차이를 검증 클립에서 확인합니다. 결과 폴더 전체를 장치로 복사하면 PyTorch 없이 `numpy`와 `ncnn`만으로 추론합니다. `ARCH='transformer'`이면 `forward()` 전체를 ONNX 한 개로 내보냅니다(ncnn 미지원).


In [ ]:
# 11. 배포용 내보내기 + 런타임 확인
needed = ('onnx', 'onnxscript', 'onnxruntime') + (('ncnn', 'pnnx') if ARCH == 'lite' else ())
missing = [name for name in needed if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
export_target = EXPERIMENT_DIR / (('deploy_' if ARCH == 'lite' else 'model_') + uuid.uuid4().hex[:8] + ('' if ARCH == 'lite' else '.onnx'))
script = 'export_lite.py' if ARCH == 'lite' else 'export_onnx.py'
run_python(script, '--checkpoint', RUN_DIR / 'best.pt', '--output', export_target,
           '--check-input', DATA_DIR / chosen['file'])
print('내보내기:', export_target)


## 저장 위치 및 다시 시작하기
- `MyDrive/GigaHands/runs/<RUN_NAME>/checkpoints/best.pt`: 검증 오차가 가장 낮은 가중치
- `last.pt`: 마지막 완료 epoch의 모델·optimizer·scheduler·난수 상태
- `run.json`, `metrics.jsonl`: 설정과 학습 기록
- 상위 실행 폴더: 소스 사본, 학습 곡선, 기준선 비교, 예측 결과, 배포용 모델(`deploy_*/` 또는 `model_*.onnx`)

Colab 재연결 시 위에서부터 실행하고 같은 `RUN_NAME`과 설정을 사용하세요. GPU smoke test는 생략할 수 있습니다. 7번 셀이 `last.pt`를 찾아 자동 재개합니다.

**실행 검증 범위:** 노트북 구조·내장 코드·데이터 검사·학습/재개/평가/추론 파이프라인은 로컬 CPU로 검증했습니다. 이 파일을 만들 때 실제 Colab GPU 세션에서 전체 학습을 수행한 것은 아닙니다. Colab에서 6번 GPU smoke test로 CUDA 경로를 먼저 확인하세요.
